In [1]:
import os
import idaes.logger as idaeslog
from pyomo.environ import (
    ConcreteModel,
    Constraint,
    Var,
    Objective,
    Expression,
    value,
    TransformationFactory,
    units as pyunits, 
    Block,
)
from pyomo.network import Arc, SequentialDecomposition
from pyomo.util.check_units import assert_units_consistent

from idaes.core import (
    FlowsheetBlock, 
    MomentumBalanceType, 
    UnitModelCostingBlock,
    UnitModelBlockData,
)
from watertap.core.solvers import get_solver
from idaes.core.util.initialization import (
    propagate_state,
    fix_state_vars,
    revert_state_vars,
)
from idaes.core.util.exceptions import ConfigurationError
from idaes.models.unit_models.translator import Translator
from idaes.models.unit_models import (
    Mixer, 
    Heater,
    Separator,
    Product,
    Feed,
    PressureChanger,
)
from idaes.models.unit_models.mixer import MomentumMixingType, MixingType
from idaes.models.unit_models.separator import (
    SplittingType,
    EnergySplittingType, 
)
import idaes.core.util.scaling as iscale
import pandas as pd 
from watertap.core.util.initialization import assert_degrees_of_freedom, check_solve
from watertap.property_models.multicomp_aq_sol_prop_pack import (
    MCASParameterBlock,
    DiffusivityCalculation,
    MaterialBalanceType,
    ElectricalMobilityCalculation,
)
from watertap.core.wt_database import Database
import watertap.core.zero_order_properties as prop_ZO
from watertap.unit_models.zero_order import (
    FeedZO,
    SWOnshoreIntakeZO,
    WaterPumpingStationZO,
    ScreenZO,
    ChemicalAdditionZO,
    ChlorinationZO,
    StaticMixerZO,
    CoagulationFlocculationZO,
    SedimentationZO,
    AirFlotationZO,
    FixedBedZO,
    StorageTankZO,
    IonExchangeZO,
    TriMediaFiltrationZO,
    BackwashSolidsHandlingZO,
    CartridgeFiltrationZO,
    GACZO,
    LandfillZO,
)
from watertap.unit_models.nanofiltration_ZO import(
    NanofiltrationZO,
    MaterialBalanceType,
    EnergyBalanceType,
    MomentumBalanceType,
)
from watertap.unit_models.electrodialysis_0D import (
    Electrodialysis0D,
    MaterialBalanceType,
    EnergyBalanceType,
    MomentumBalanceType,
)
from watertap.unit_models.electrodialysis_1D import (
    ElectricalOperationMode,
    PressureDropMethod,
    FrictionFactorMethod,
    HydraulicDiameterMethod,
    LimitingCurrentDensityMethod,
)
from watertap.unit_models.electrodialysis_1D import Electrodialysis1D
from watertap.unit_models.pressure_exchanger import PressureExchanger
from watertap.unit_models.pressure_changer import Pump, EnergyRecoveryDevice

from watertap.unit_models.electrolyzer import Electrolyzer

from watertap.costing.zero_order_costing import ZeroOrderCosting
from watertap.costing import WaterTAPCosting
from idaes.core.util.misc import StrEnum

class XLocation(StrEnum):
    offshore = "offshore"
    onshore = "onshore"
    no_location= "no_location"

class ERDType1(StrEnum):
    pressure_exchanger = "pressure_exchanger"
    pump_as_turbine = "pump_as_turbine"
    no_ERD = "no_ERD"

class ERDType2(StrEnum):
    pressure_exchanger = "pressure_exchanger"
    pump_as_turbine = "pump_as_turbine"
    no_ERD = "no_ERD"

class ELECType(StrEnum):
    AWE = "AWE" #Alkaline Water Electrolysis 
    PEM = "PEM" #Proton Exchange Membrane Electrolysis
    no_ELEC = "no_ELEC"

def x_location_not_found(x_location):
    raise NotImplementedError(
        "x_location was {}, but can only be offshore, onshore, or no_location"
        "".format(x_location.value)
    )
def erd_type1_not_found(erd_type1):
    raise NotImplementedError(
        "erd_type1 was {}, but can only be pressure_exchanger, pump_as_turbine, or no_ERD"
        "".format(erd_type1.value)
    )

def erd_type2_not_found(erd_type2):
    raise NotImplementedError(
        "erd_type2 was {}, but can only be pressure_exchanger, pump_as_turbine, or no_ERD"
        "".format(erd_type2.value)
    )

def elec_type_not_found(elec_type):
    raise NotImplementedError(
        "elec_type was {}, but can only be AWE, PEM, or no_ELEC"
        "".format(elec_type.value)
    )

# Set up logger
_log = idaeslog.getLogger(__name__)

def build_flowsheet(x_location=XLocation.offshore, erd_type1=ERDType1.pressure_exchanger, erd_type2=ERDType2.pressure_exchanger, elec_type=ELECType.AWE):
    m = build(x_location=x_location, erd_type1=erd_type1,erd_type2=erd_type2,elec_type=elec_type)
    set_operating_conditions(m)
    assert_degrees_of_freedom(m, 0)
    return m

def solve_flowsheet(flowsheet=None):
    m = flowsheet.parent_block() 
    initialize_system(m)
    assert_degrees_of_freedom(m, 0)
    optimize_operation(m) #unfixes specific variables for cost optimization
    solve(m, checkpoint="solve flowsheet after initializing system")
    display_results(m)
    add_costing(m)
    initialize_costing(m)
    assert_degrees_of_freedom(m, 0)
    solve(m, checkpoint="solve flowsheet with costing")

def main(x_location="onshore", erd_type1="pressure_exchanger", erd_type2="pressure_exchanger", elec_type= "PEM"):
    m = build_flowsheet(x_location=x_location, erd_type1=erd_type1, erd_type2=erd_type2, elec_type=elec_type)

    initialize_system(m)
    assert_degrees_of_freedom(m, 0)

    solve(m, checkpoint=f" solve flowsheet after initializing {x_location, erd_type1, erd_type2, elec_type} system")
    display_results(m)
    
    add_costing(m)
    initialize_costing(m)
    assert_degrees_of_freedom(m,0) #ensure problem is square 

    optimize_operation(m) #unfixes specific variables for cost optimization

    solve(m, tee=True, checkpoint=f" solve {x_location, erd_type1, erd_type2, elec_type} flowsheet with costing")
    display_costing(m)

    return m

def build(x_location=None, erd_type1=None, erd_type2=None, elec_type=None):
    # flowsheet set up
    m = ConcreteModel()
    m.db = Database()
    m.x_location = x_location
    m.erd_type1 = erd_type1
    m.erd_type2 = erd_type2
    m.elec_type = elec_type

    m.fs = FlowsheetBlock(dynamic=False)
    m.fs.prop_prtrt = prop_ZO.WaterParameterBlock(solute_list=["Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v","tss"])
    m.fs.prop_Mgxstor = prop_ZO.WaterParameterBlock(solute_list=["Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v"])
    m.fs.prop_Lixstor = prop_ZO.WaterParameterBlock(solute_list=["Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v"])
    m.fs.prop_psttrt = prop_ZO.WaterParameterBlock(solute_list=["tds","H_+", "OH_-","O2-v","H2-v"])
    m.fs.prop_H2stor = prop_ZO.WaterParameterBlock(solute_list=["H_+","OH_-","O2-v","H2-v"])
    m.fs.prop_od = MCASParameterBlock(material_flow_basis="mass",
                                        ignore_neutral_charge=True,
                                        solute_list=["Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v"],
                                        diffusivity_data={
                                            ("Liq", "Li_+"): 1.03e-9,
                                            ("Liq", "Ca_2+"): 7.93e-10,
                                            ("Liq", "Mg_2+"): 7.05e-10,
                                            ("Liq", "Cl_-"): 2.03e-9,
                                            ("Liq", "SO4_2-"): 1.07e-9,
                                            ("Liq", "Na_+"): 1.33e-9,
                                            ("Liq", "H_+"): 9.13e-9,
                                            ("Liq", "OH_-"): 5.27e-9,
                                            ("Liq", "O2-v"): 2e-9,
                                            ("Liq", "H2-v"): 4.58e-9
                                        },
                                        mw_data={
                                            "H2O": 0.018, 
                                            "Li_+": 0.007, 
                                            "Ca_2+": 0.040, 
                                            "Mg_2+": 0.024,
                                            "Cl_-": 0.035,
                                            "SO4_2-": 0.096,
                                            "Na_+": 0.023,
                                            "H_+": 0.001,
                                            "OH_-": 0.017,
                                            "O2-v": 0.032,
                                            "H2-v":0.002
                                        },
                                        elec_mobility_data={
                                            "Li_+": 4.08e-8,
                                            "Ca_2+": 6.28e-8, 
                                            "Mg_2+": 5.58e-8,
                                            "Cl_-": 8.04e-8,
                                            "SO4_2-": 8.47e-8,
                                            "Na_+": 5.26e-8,
                                            "H_+": 3.68e-7 ,
                                            "OH_-": 2.05e-7
                                        },
                                        charge={
                                            "Li_+": 1,
                                            "Ca_2+": 2,
                                            "Mg_2+": 2,
                                            "Cl_-": -1,
                                            "SO4_2-": -2,
                                            "Na_+": 1,
                                            "H_+": 1,
                                            "OH_-":-1
                                        },
                                        diffus_calculation=DiffusivityCalculation.none,
                                        elec_mobility_calculation=ElectricalMobilityCalculation.none,
                                        )
    
    # Build the state block and specify a time (0 = steady state).
    m.fs.state_block = m.fs.prop_od.build_state_block([0])

    # Specify the state variables of the stream. Note, now we specify mass flowrate (`flow_mass_phase_comp`) instead of molar flowrate (`flow_mol_phase_comp`).
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "Li_+"].fix(1.7e-7)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "Ca_2+"].fix(0.0004)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "Mg_2+"].fix(0.00128)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "Cl_-"].fix(0.0194)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "SO4_2-"].fix(0.0027)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "Na_+"].fix(0.0108)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "H_+"].fix(0.06)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "OH_-"].fix(0.0000214)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "O2-v"].fix(0.477)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "H2-v"].fix(0.12)
    m.fs.state_block[0].flow_mass_phase_comp["Liq", "H2O"].fix(0.926)
    m.fs.state_block[0].pressure.fix(101325)
    m.fs.state_block[0].temperature.fix(293.15)

    # "Touch" build-on-demand variables so that they are created
    m.fs.state_block[0].flow_mass_phase_comp
    m.fs.state_block[0].flow_mol_phase_comp
    #m.fs.state_block[0].dens_mass_phase_comp
    m.fs.state_block[0].conc_mol_phase_comp
    m.fs.state_block[0].flow_vol_phase
    m.fs.state_block[0].molality_phase_comp
    m.fs.state_block[0].conc_mass_phase_comp
    m.fs.state_block[0].total_hardness
    m.fs.state_block[0].ionic_strength_molal
    m.fs.state_block[0].enth_mass_phase
    m.fs.state_block[0].pressure_osm_phase


    density= 1023.5 * pyunits.kg/ pyunits.m**3
    m.fs.prop_prtrt.dens_mass_default = density

    #block structure
    prtrt = m.fs.pretreatment = Block()
    Mgx = m.fs.Mgextraction = Block()
    Mgxstor = m.fs.Mgstorage = Block ()
    Lix = m.fs.Liextraction = Block()
    Lixstor =m.fs.Listorage = Block()
    desal = m.fs.desalination = Block()
    psttrt = m.fs.posttreatment = Block()
    H2x = m.fs.H2extraction = Block()
    H2xstor = m.fs.H2storage = Block()

    #Unit models
    m.fs.feedzo = FeedZO(property_package=m.fs.prop_prtrt)
  
    #Pretreatment
    if x_location =="onshore":
        prtrt.intake = SWOnshoreIntakeZO(property_package=m.fs.prop_prtrt, database=m.db)
    elif x_location == "offshore":
        prtrt.intake = WaterPumpingStationZO(property_package=m.fs.prop_prtrt, database=m.db, process_subtype="raw")
    else: 
        raise ConfigurationError(
            "extraction location, X_location"
            "was {}, but can only be"
            "onshore or offshore"
            "".format(x_location)
        )
    prtrt.ferric_chloride_addition = ChemicalAdditionZO(property_package=m.fs.prop_prtrt, database=m.db, process_subtype="ferric_chloride")
    prtrt.chlorination = ChlorinationZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.static_mixer = StaticMixerZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.storage = StorageTankZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.screening = ScreenZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.coag_and_floc = CoagulationFlocculationZO(
        property_package=m.fs.prop_prtrt, database=m.db
    )
    prtrt.sedimentation = SedimentationZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.flotation = AirFlotationZO(property_package=m.fs.prop_prtrt, database=m.db, )
    prtrt.gravity_basin = FixedBedZO(
        property_package=m.fs.prop_prtrt, database=m.db, process_subtype="gravity_basin"
    )
    prtrt.mfiltration = TriMediaFiltrationZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.mbackwash_pump =BackwashSolidsHandlingZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.antiscalant_addition = ChemicalAdditionZO(property_package=m.fs.prop_prtrt, database=m.db, process_subtype="anti-scalant")
    prtrt.cfiltration = CartridgeFiltrationZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.gac = GACZO(property_package=m.fs.prop_prtrt, database=m.db, process_subtype="pressure_vessel")
    prtrt.gbackwash_pump = BackwashSolidsHandlingZO(property_package=m.fs.prop_prtrt, database=m.db)

    prtrt.mf_disposal = LandfillZO(property_package=m.fs.prop_prtrt, database=m.db)
    prtrt.gac_disposal = LandfillZO(property_package=m.fs.prop_prtrt, database=m.db)

    #Powered Softening (Mg extraction)
    Mgx.P1 = Pump(property_package=m.fs.prop_od)
    Mgx.S1 = Separator(property_package=m.fs.prop_od,
                        outlet_list=["inlet_diluate", "inlet_concentrate"])
    Mgx.nano = NanofiltrationZO(property_package=m.fs.prop_od,
                                 material_balance_type=MaterialBalanceType.useDefault,
                                 energy_balance_type=EnergyBalanceType.none,
                                 momentum_balance_type=MomentumBalanceType.pressureTotal
                                 )
    Mgx.treated = Product(property_package=m.fs.prop_od)
    Mgx.byproduct = Product(property_package=m.fs.prop_od)

    Mgx.S1.inlet_diluate_state[0].conc_mass_phase_comp[...]
    Mgx.S1.inlet_concetrate_state[0].conc_mass_phase_comp[...]
    Mgx.treated.properties[0].conc_mass_phase_comp[...]
    Mgx.byproduct.properties[0].conc_mass_phase_comp[...]

    Mgx.S1.inlet_diluate_state[0].flow_vol_phase[...]
    Mgx.S1.inlet_concentrate_state[0].flow_vol_phase[...]
    Mgx.treated.properties[0].flow_vol_phase[...]
    Mgx.byproduct.properties[0].flow_vol_phase[...]

    #Magnesium Storage & Waste Disposal
    Mgxstor.storage = StorageTankZO(property_package=m.fs.prop_Mgxstor, database=m.db)
    Mgxstor.disposal = BackwashSolidsHandlingZO(property_package=m.fs.prop_Mgxstor, database=m.db)
    Mgxstor.landfill = LandfillZO(property_package=m.fs.prop_Mgxstor, database=m.db)


    #Powered Li extraction (Li+ Electrodialysis)
    Lix.P1 = Pump(property_package=m.fs.prop_od)
    Lix.S1 = Separator(property_package=m.fs.prop_od,
                       outlet_list=["inlet_diluate", "inlet_concentrate"],
                       )
    Lix.Elec = Electrodialysis0D(
            property_package=m.fs.prop_od,
            material_balance_type=MaterialBalanceType.useDefault,
            energy_balance_type=EnergyBalanceType.none,
            momentum_balance_type=MomentumBalanceType.pressureTotal,
            )
    Lix.product = Product(property_package=m.fs.prop_od)
    Lix.treated = Product(property_package=m.fs.prop_od)

    Lix.S1.inlet_diluate_state[0].conc_mass_phase_comp[...]
    Lix.S1.inlet_concentrate_state[0].conc_mass_phase_comp[...]
    Lix.product.properties[0].conc_mass_phase_comp[...]
    Lix.treated.properties[0].conc_mass_phase_comp[...]

    Lix.S1.inlet_diluate_state[0].flow_vol_phase[...]
    Lix.S1.inlet_concentrate_state[0].flow_vol_phase[...]
    Lix.product.properties[0].flow_vol_phase[...]
    Lix.treated.properties[0].flow_vol_phase[...]
    
    #Reporting variables 
    Lix.mem_area = Var(
        initialize=1,
        boud=(0, 1e3),
        units=pyunits.meter**2,
        doc="Total membrane area for cem (or aem) in one stack",
    )

    Lix.product_concentration = Var(
        initialize=1, bounds=(0, 1000), units=pyunits.kg * pyunits.meter**-3
    )

    Lix.treated_concentration = Var(
        initialize=1, bounds=(0, 1e6), units=pyunits.kg * pyunits.meter**-3
    )

    #Reporting variable constraints
    Lix.eq_product_concentration = Constraint(
        expr=Lix.product_concentration
        == Lix.product.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
    )
    Lix.eq_treated_concentration = Constraint(
        expr=Lix.treated_concentration
        == Lix.treated.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
    )
    Lix.eq_mem_area = Constraint(
        expr=m.fs.mem_area
        == Lix.Elec.cell_width
        * Lix.Elec.cell_length
        * Lix.Elec.cell_pair_num
    )

    #Li Storage Unit
    Lixstor.storage = StorageTankZO(property_package=m.fs.prop_Lixstor, database=m.db)
    Lixstor.disposal = BackwashSolidsHandlingZO(property_package=m.fs.prop_Lixstor, database=m.db)
    Lixstor.landfill = LandfillZO(property_package=m.fs.prop_Lixstor, database=m.db)
    
    #Electrodialysis desalination
    desal.S0 = Separator(
        property_package=m.fs.prop_od,
        outlet_list=["to_dil_in", "to_conc_in0"],
    )
    desal.M0 = Mixer(
        property_package=m.fs.prop_od,
        energy_mixing_type=MixingType.none,
        momentum_mixing_type=MomentumMixingType.none,
        inlet_list=["from_feed", "from_conc_out"],
    )
    desal.P0 = Pump(property_package=m.fs.prop_od)
    desal.P0.ratioP_calculation.deactivate()
    desal.P1 = Pump(property_package=m.fs.prop_od)
    desal.P0.ratioP_calculation.deactivate()

    desal.Elec = Electrodialysis1D(
        property_package=m.fs.prod_od,
        operation_mode=ElectricalOperationMode.Constant_Voltage,
        finite_elements=20,
        has_pressure_change=True,
        has_nonohmic_potential_membrane=True,
        has_Nernst_diffusion_layer=True,
        limiting_current_density_method=LimitingCurrentDensityMethod.Theoretical,
        pressure_drop_method=PressureDropMethod.Darcy_Weisbach,
        hydraulic_diameter_method=HydraulicDiameterMethod.spacer_specific_area_known,
        firction_factor_method=FrictionFactorMethod.Gurreri,
    )
    desal.S1 = Separator(
        property_package=m.fs.prop_od,
        outlet_list=["to_ERD", "to_conc_in1"],
    )
    desal.ERD = EnergyRecoveryDevice(property_package=m.fs.prop_od, )
    desal.product = Product(property_package=m.fs.prop_od)
    desal.disposal = Product(property_package=m.fs.prop_od)

    desal.product.properties[0].flow_vol_phase[...]
    desal.disposal.properties[0].flow_vol_phase[...]

    desal.feed_salinity = Var(
        intialize=1,
        domain=NonNegativeReals,
        units=pyunits.kg / pyunits.m**3,
        doc="Salinity of feed solution",
    )

    desal.eq_recovery_vol_H2O = Constraint(
        expr=desal.recovery_vol_H2O * desal.Elec.oulet_diluate.properties[0].flow_vol_phase["Liq"]
        == desal.product.properties[0].flow_vol_phase["Liq"]
    )

    desal.eq_electrodialysis_equal_flow = Constraint(
        expr=desal.Elec.diluate.properties[0,0].flow_vol_phase["Liq"]
        == desal.Elec.concentrate.properties[0,0].flow_vol_phase["Liq"]
    )

    desal.eq_feed_salinity = Constraint(
        expr=desal.feed_salinity
        == (desal.S0.inlet.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
            + desal.S0.inlet.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
            + desal.S0.inlet.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
            + desal.S0.inlet.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
            + desal.S0.inlet.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
        )
    )

    desal.product_salinity = Constraint(
        expr= (desal.product.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                + desal.product.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                + desal.product.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                + desal.product.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                + desal.product.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
        )
    )

    desal.disposal_salinity = Constraint(
        expr= (desal.disposal.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                + desal.disposal.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                + desal.disposal.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                + desal.disposal.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                + desal.disposal.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
        )
    )

    desal.mem_area = Expression(
        expr=desal.Elec.cell_width
        * desal.Elec.cell_length
        * desal.Elec.cell_pair_num
    )

    desal.voltage_per_cp = Expression(
        expr=desal.Elec.voltage_applied[0] / desal.Elec.cell_pair_num
    )

    #Demineralization 
    psttrt.IX = IonExchangeZO(property_package=m.fs.prop_psttrt, database=m.db, process_subtype="demineralization")
    psttrt.storage = StorageTankZO(property_package=m.fs.prop_psttrt, database=m.db)
    psttrt.disposal = BackwashSolidsHandlingZO(property_package=m.fs.prop_psttrt, database=m.db)
    psttrt.landfill = LandfillZO(property_package=m.fs.prop_psttrt, database=m.db)

    #H2 Extraction
    H2x.Heat = Heater(property_package=m.fs.prop_od)
    H2x.S1 = Separator(property_package=m.fs.prop_od,
                        outlet_list=["inlet_cathode", "inlet_anode"],)
    H2x.S1.inlet_cathode_state[0].conc_mass_phase_comp[...]
    H2x.S1.inlet_anode_state[0].conc_mass_phase_comp[...]
    H2x.S1.inlet_cathode_state[0].flow_vol_phase[...]
    H2x.S1.inlet_anode_state[0].flow_vol_phase[...]

    if elec_type == "PEM":
        H2x.ER = Electrolyzer(property_package=m.fs.prop_od)
        #Touch electrolysis properties
        H2x.ER.anolyte.properties_in[0].flow_vol_phase
        H2x.ER.anolyte.properties_in[0].conc_mass_phase_comp
        H2x.ER.anolyte.properties_in[0].conc_mol_phase_comp
        H2x.ER.catholyte.properties_in[0].flow_vol_phase
        H2x.ER.catholyte.properties_in[0].conc_mass_phase_comp
        H2x.ER.catholyte.properties_in[0].conc_mol_phase_comp
        H2x.ER.anolyte.properties_out[0].flow_vol_phase
        H2x.ER.anolyte.properties_out[0].conc_mass_phase_comp
        H2x.ER.anolyte.properties_out[0].conc_mol_phase_comp
        H2x.ER.catholyte.properties_out[0].flow_vol_phase
        H2x.ER.catholyte.properties_out[0].conc_mass_phase_comp
        H2x.ER.catholyte.properties_out[0].conc_mol_phase_comp


        H2x.Compressor_O2 = PressureChanger(
            property_package=m.fs.prop_od,
            compressor = True,
        )
        H2x.Compressor_H2 = PressureChanger(
            property_package=m.fs.prop_od,
            compressor = True,

        )
        H2x.prod_O2 = Product(property_package=m.fs.prop_od)
        H2x.prod_O2.properties[0].conc_mass_phase_comp[...]
        H2x.prod_O2.properties[0].flow_vol_phase[...]
        H2x.prod_H2 = Product(property_package=m.fs.prop_od)
        H2x.prod_H2.properties[0].conc_mass_phase_comp[...]
        H2x.prod_H2.properties[0].flow_vol_phase[...]
    elif elec_type == "AWE":
        H2x.ER = Electrolyzer(property_package=m.fs.prop_od)
        #Touch electrolysis properties
        H2x.ER.anolyte.properties_in[0].flow_vol_phase
        H2x.ER.anolyte.properties_in[0].conc_mass_phase_comp
        H2x.ER.anolyte.properties_in[0].conc_mol_phase_comp
        H2x.ER.catholyte.properties_in[0].flow_vol_phase
        H2x.ER.catholyte.properties_in[0].conc_mass_phase_comp
        H2x.ER.catholyte.properties_in[0].conc_mol_phase_comp
        H2x.ER.anolyte.properties_out[0].flow_vol_phase
        H2x.ER.anolyte.properties_out[0].conc_mass_phase_comp
        H2x.ER.anolyte.properties_out[0].conc_mol_phase_comp
        H2x.ER.catholyte.properties_out[0].flow_vol_phase
        H2x.ER.catholyte.properties_out[0].conc_mass_phase_comp
        H2x.ER.catholyte.properties_out[0].conc_mol_phase_comp


        H2x.Compressor_O2 = PressureExchanger(
            property_package=m.fs.prop_od,
            compressor = True,
        )
        H2x.Compressor_H2 = PressureExchanger(
            property_package=m.fs.prop_od,
            compressor = True,

        )
        H2x.prod_O2 = Product(property_package=m.fs.prop_od)
        H2x.prod_O2.properties[0].conc_mass_phase_comp[...]
        H2x.prod_H2 = Product(property_package=m.fs.prop_od)
        H2x.prod_H2.properties[0].conc_mass_phase_comp[...]  

    else: 
        pass 

    #Gas Storage
    H2xstor.O2 = StorageTankZO(property_package=m.fs.prop_prtrt, database=m.db, process_subtype="default")
    H2xstor.H2 = StorageTankZO(property_package=m.fs.prop_prtrt, database=m.db, process_subtype="default")

    #Translator Blocks

    #Pretreatment --> Mg extraction
    m.fs.tb_prtrt_Mgx = Translator(
        inlet_property_package=m.fs.prop_prtrt, outlet_property_package=m.fs.prop_od
    )
    @m.fs.tb_prtrt_Mgx.Constraint(["H2O","Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v","tss"])
    def eq_flow_mass_comp(blk, j):
        if j == "tss":
            return ( 
                blk.properties_in[0].flow_mass_comp["tss"]
                + blk.properties_in[0].flow_mass_comp["Mg_2+"]
                == blk.properties_out[0].flow_mass_phase_comp["Liq", "Mg_2+"]  
            )
        else: 
            jj = j
            return (
                blk.properties_in[0].flow_mass_comp[j]
                == blk.properties_out[0].flow_mass_phase_comp["Liq", jj]
            )
        
    # Mg extraction --> Mg storage
    m.fs.tb_Mgx_Mgxstor = Translator (
        inlet_property_package=m.fs.prop_od, outlet_property_package=m.fs.prop_Mgxstor
    )
    @m.fs.tb_Mgx_Mgxstor.Constraint(["H2O","Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v"])
    def eq_flow_mass_comp(blk, j):
        if j == "Mg_2+":
            return ( 
                blk.properties_in[0].flow_mass_phase_comp["Liq", "Mg_2+"]
                == blk.properties_out[0].flow_mass_comp["Mg_2+"]
            ) 
        else:
            pass
        
   #Li extraction to Li storage 
    m.fs.tb_Lix_Lixstor = Translator(
        inlet_property_package=m.fs.prop_od, outlet_property_package=m.fs.prop_Lixstor
    )
    @m.fs.tb_Lix_Lixstor.Constraint(["H2O","Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v"])
    def eq_flow_mass_comp(blk, j):
        if j == "Li_+":
            return (
                blk.properties_in[0].flow_mass_phase_comp["Liq", "Li_+"]
                == blk.properties_out[0].flow_mass_comp["Li_+"]
            )
        else: 
           jj = j 
           return (
               blk.properties_in[0].flow_mass_phase_comp["Liq", j]
               == blk.properties_out[0].flow_mass_comp[jj]
           )
    
    
     #Desalination --> Posttreatment
    m.fs.tb_desal_psttrt = Translator(
        inlet_property_package=m.fs.prop_od, outlet_property_package=m.fs.prop_psttrt
    )
    @m.fs.tb_desal_psttrt.Constraint(["H2O","Li_+","Ca_2+","Mg_2+","Cl_-","SO4_2-","Na_+","H_+","OH_-","O2-v","H2-v", "tds"])
    def eq_flow_mass_comp(blk, j):
        if j == "Li_+" or "Mg_2+" or "Cl_-" or "SO4_2-" or "Na_+":
            return (
                blk.properties_in[0].flow_mass_phase_comp["Liq", "Li_+"]
                + blk.properties_in[0].flow_mass_phase_comp["Liq", "Ca_2+"]
                + blk.properties_in[0].flow_mass_phase_comp["Liq", "Mg_2+"]
                + blk.properties_in[0].flow_mass_phase_comp["Liq", "Cl_-"]
                + blk.properties_in[0].flow_mass_phase_comp["Liq", "SO4_2-"]
                + blk.properties_in[0].flow_mass_phase_comp["Liq", "Na_+"]
                == blk.properties_out[0].flow_mass_comp["tds"]
            )
        else:
            jj = j 
            return (
            blk.properties_in[0].flow_mass_phase_comp["Liq", j]
            == blk.properties_out[0].flow_mass_comp[jj]
            )

    #Posttreatment --> H2 Extraction 
    m.fs.tb_psttrt_H2x = Translator(
        inlet_property_package=m.fs.prop_psttrt, outlet_property_package=m.fs.prop_od
    )
    @m.fs.tb_psttrt_H2x.Constraint(["H2O","H_+", "OH_-","O2-v","H2-v"])
    def eq_flow_mass_comp(blk, j):
        if j == "H_+":
            return (
                blk.properties_in[0].flow_mass_comp["H_+"]
                == blk.properties_out[0].flow_mass_phase_comp["Liq", "H_+"]
            )
        else: 
            jj = j 
        return (
            blk.properties_in[0].flow_mass_comp[j]
            == blk.properties_out[0].flow_mass_phase_comp[jj]
        )
    
    #H2 Extraction --> H2 Gas Storage
    m.fs.tb_H2x_H2stor = Translator (
        inlet_property_package=m.fs.prop_od, outlet_property_package=m.fs.prop_zo
    )
    @m.fs.tb_H2x_H2stor.Constraint(["H2-v"])
    def eq_flow_mass_comp_H2(blk, j):
        if j == "H2-v":
            return (
                blk.properties_in[0].flow_mass_phase_comp["Liq", "H2-v"]
                == blk.properties_out[0].flow_mass_comp["H2-v"]
            )
        else:
            pass
    #H2 Extraction -->  O2 Gas Storage
    m.fs.tb_H2x_O2stor = Translator (
        inlet_property_package=m.fs.prop_od, outlet_property_package=m.fs.prop_zo
    )
    @m.fs.tb_H2x_O2stor.Constraint(["O2-v"])
    def eq_flow_mass_comp_O2(blk, j):
        if j == "O2-v":
            return (
                blk.properties_in[0].flow_mass_phase_comp["Liq", "O2-v"]
                == blk.properties_out[0].flow_mass_comp["O2-v"]
            )
        else: 
            pass

    #Connections
    #Pretreatment
    m.fs.s_feed = Arc(source=m.fs.feed.outlet, destination=prtrt.intake.inlet)
    prtrt.s01 = Arc(sourced=prtrt.intake.outlet, destination=prtrt.ferric_chloride_addition.inlet)
    prtrt.s02 = Arc(source=prtrt.ferric_chloride_addition.outlet, destination=prtrt.chlorination.inlet)
    prtrt.s03 = Arc(source=prtrt.chlorination.outlet , destination=prtrt.static_mixer.inlet)
    prtrt.s04 = Arc(source=prtrt.static_mixer.outlet, destination=prtrt.storage.inlet)
    prtrt.s05 = Arc(source=prtrt.storage.outlet, destination=prtrt.screening.inlet)
    prtrt.s06 = Arc(source=prtrt.screening.treated, destination=prtrt.coag_and_floc.inlet)
    prtrt.s06 = Arc(source=prtrt.coag_and_floc.outlet, destination=prtrt.sedimentation.inlet)
    prtrt.s07 = Arc(source=prtrt.sedimentation.treated, destination=prtrt.flotation.inlet)
    prtrt.s08 = Arc(source=prtrt.flotation.treated, destination=prtrt.gravity_basin.inlet)
    prtrt.s09 = Arc(source=prtrt.gravity_basin.treated, destination=prtrt.mfiltration.inlet)
    prtrt.s10 = Arc(source=prtrt.mfiltration.byproduct, destination=prtrt.mbackwash_pump.inlet)
    prtrt.s11 = Arc(source=prtrt.mfiltration.treated, destination=prtrt.antiscalant_addition.inlet)
    prtrt.s12 = Arc(source=prtrt.antiscalant_addition.outlet, destination=prtrt.cfiltration.inlet)
    prtrt.s13 = Arc(source=prtrt.cfiltration.treated, destination=prtrt.gac.inlet)
    prtrt.s14 = Arc(source=prtrt.gac.byproduct, destination=m.fs.gbackwash_pump.inlet)
   
    m.fs.mlandfill = Arc(source=prtrt.mbackwash_pump.byproduct, destination=prtrt.mf_disposal.inlet)
    m.fs.glandfill = Arc(source=prtrt.gbackwash_pump.byproduct, destination=prtrt.gac_disposal.inlet)  

   #Pretreatment --> Powered Softening (Magnesium Nanofiltration)
    m.fs.s_prtrt_tb = Arc(source=prtrt.gac.treated, destination=m.fs.tb_prtrt_Mgx.inlet)

    #Powered Softening (Magnesium Nanofiltration)
    m.fs.s_tb_Mgx = Arc(source=m.fs.tb_prtrt_Mgx.outlet, destination=Mgx.P1.inlet)
    Mgx.s01 = Arc(source=Mgx.P1.outlet, destination=Mgx.S1.inlet)
    Mgx.s02 = Arc(source=Mgx.S1.inlet_diluate, destination=Mgx.nano.inlet_diluate)
    Mgx.s03 = Arc(source=Mgx.S1.inlet_concentrate, destination=Mgx.nano.inlet_concentrate)
    Mgx.s04 = Arc(source=Mgx.nano.outlet_diluate, destination=Mgx.treated.inlet)
    Mgx.s05 = Arc(source=Mgx.nano.outlet_concentrtrate, destination=Mgx.byproduct.inlet)

    #Magnesium Extraction --> Magnesium Storage
    m.fs.s_Mgx_tb = Arc(source=m.fs.tb_Mgx_Mgxstor.outlet, destination=Mgxstor.storage.inlet)
    Mgxstor.s01 = Arc(source=Mgxstor.storage.outlet, destination=Mgxstor.disposal.inlet)
    Mgxstor.s02 = Arc(source=Mgxstor.disposal.outlet, destination=Mgxstor.landfill.inlet)

    #Powered Lithium Extraction (Li_+ Electrodialysis)
    Lix.s01 = Arc(source=Mgx.treated.inlet, destination=Lix.P1.inlet)
    Lix.s02 = Arc(source=Lix.P1.outlet, destination=Lix.S1.inlet)
    Lix.s03 = Arc(source=Lix.S1.inlet_diluate, destination=Lix.Elec.inlet_diluate)
    Lix.s04 = Arc(source=Lix.S1.inlet_concentrate, destination=Lix.Elec.inlet_concentrate)
    Lix.s05 = Arc(source=Lix.Elec.outlet_diluate, destination=Lix.treated.inlet)
    Lix.s06 = Arc(souce=Lix.Elec.outlet_concentrate, destination=Lix.product.inlet)
    
    #Lithium Extraction Product --> Lithium Storage
    m.fs.s_Lix_tb = Arc(source=Lix.product.inlet, destination=m.fs.tb_Lix_Lixstor.inlet)

    #Lithium Storage 
    m.fs.s_tb_Lixstor = Arc(source=m.fs.tb_Lix_Lixstor.outlet, destination=Lixstor.storage.inlet)
    Lixstor.s01 = Arc(source=Lixstor.storage.outlet, destination=Lixstor.disposal.inlet)
    Lixstor.s02 = Arc(source=Lixstor.disposal.outlet, destination=Lixstor.landfill.inlet)

  
    #Double-Pass Electrodialysis Desalination 
    desal.s01 = Arc(source=Lix.treated.inlet, destination=desal.S0.inlet)
    desal.s02 = Arc(source=desal.S0.to_dil_in, destination=desal.P1.inlet)
    desal.s03 = Arc(source=desal.P1.outlet, destination=desal.Elec.inlet_diluate)
    desal.s04 = Arc(source=desal.S0.to_conc_in0, destination=desal.M0.from_feed)
    desal.s05 = Arc(source=desal.M0.outlet, destination=desal.P0.inlet)
    desal.s06 = Arc(source=desal.P0.outlet, destination=desal.Elec.inlet_concentrate)
    desal.s07 = Arc(source=desal.Elec.outlet_diluate, destination=desal.product.inlet)
    desal.s08 = Arc(source=desal.Elec.outlet_concentrate, destination=desal.S1.inlet)
    desal.s09 = Arc(source=desal.S1.to_ERD, destination=desal.ERD.inlet)
    desal.s10 = Arc(source=desal.ERD.outlet, destination=desal.disposal.inlet)
    desal.s10 = Arc(source=desal.S1.to_conc_in1, destination=desal.M0.from_conc_out)

    #Desalination --> Posttreatment
    m.fs.s_desal_tb = Arc(source=desal.product.inlet, destination=m.fs.tb_desal_psttrt.inlet)
    
    #Posttreatment
    m.fs.s_tb_psttrt = Arc(source=m.fs.tb_desal_psttrt.outlet, destination=psttrt.IX.inlet)
    psttrt.s01 = Arc(source=psttrt.IX.treated, destination=psttrt.storage.inlet)
    psttrt.s02 = Arc(source=psttrt.storage.outlet, destination=psttrt.disposal.inlet)
    psttrt.s03 = Arc(source=psttrt.disposal.outlet, destination=psttrt.landfill.inlet)

    #Posttreatment to H2 Extraction
    m.fs.s_psttrt_tb = Arc(source=psttrt.storage.outlet, destination=m.fs.tb_psttrt_H2x.inlet)

    #H2 Extraction need to add splitter 
    m.fs.s_tb_H2 = Arc(source=m.fs.tb_psttrt_H2x.outlet, destination=H2x.Heat.inlet)
    H2x.s01 = Arc(source=H2x.Heat.outlet, destination=H2x.S1.inlet)
    H2x.s02 = Arc(source=H2x.S1.inlet_cathode, destination=H2x.ER.catholyte_inlet)
    H2x.s03 = Arc(source=H2x.S1.inlet_anode, destination=H2x.ER.anolyte_inlet)
    H2x.s04 = Arc(source=H2x.ER.catholyte_outlet, destination=H2x.Compressor_H2.inlet)
    H2x.s05 = Arc(source=H2x.ER.anolyte_outlet, destination=H2x.Compressor_O2.inlet)
    H2x.s06 = Arc(source=H2x.Compressor_H2.outlet, destination=H2x.prod_H2.inlet)
    H2x.s07 = Arc(source=H2x.Compressor_O2.outlet, destination=H2x.prod_O2.inlet)

    #H2 Extraction --> Gas Storage
    m.fs.s_H2_tb = Arc(source=H2x.prod_H2.inlet, destination=m.fs.tb_H2x_H2xstor.inlet)
    m.fs.s_O2_tb = Arc(source=H2x.prod_O2.inlet, destination=m.fs.tb_H2x_H2xstor.inlet)

    #Gas Storage 
    m.fs.s_tb_H2xstor_H2= Arc(source=m.fs.tb_H2x_H2xstor.outlet, destination=H2xstor.H2.inlet)
    m.fs_s_tb_H2xstor_O2 = Arc(source=m.fs.tb_H2x_H2xstor.outlet, destination=H2xstor.O2.inlet)

    TransformationFactory("network.expand_arcs").apply_to(m)

    #scaling
    #Set default property values
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e4, index=("Liq", "Li_+"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "Ca_2+"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e3, index=("Liq", "Mg_2+"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "Cl_-"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e3, index=("Liq", "SO4_2-"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "Na_+"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "H_+"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e5, index=("Liq", "OH_-"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 10, index=("Liq", "O2-v"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 10, index=("Liq", "H2-v"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 10, index=("Liq", "H2O"))

    #For H2x, may not be able to do this...
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1, index=("Liq", "H2O"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1, index=("Liq", "OH_-"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1, index=("Liq", "O2-v"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1, index=("Liq", "H2-v"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1, index=("Liq", "H_+"))


    #Set unit model values

    #Magnesium extraction pump
    iscale.set_scaling_factor(Mgx.P1.control_volume.work, 1e-5)

    #Magnesium extraction nanofiltration unit
    iscale.set_scaling_factor(Mgx.nano.area, 1e-1)

    #Lithium extraction pump
    iscale.set_scaling_factor(Lix.P1.control_volume.work, 1e-5) 
    
    #Lithium extraction unit
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1, index=("Liq", "H2O"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1e-3, index=("Liq", "Li_+"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1e-3, index=("Liq", "Cl_-"))
    iscale.set_scaling_factor(Lix.Elec.cell_width, 10)
    iscale.set_scaling_factor(Lix.Elec.cell_length, 10)

    #Electrodialysis recirculation desalination
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 0.1, index=("Liq", "H2O"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1e2, index=("Liq", "Na_+"))
    m.fs.prop_od.set_default_scaling("flow_mol_phase_comp", 1e2, index=("Liq", "Cl_-"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "Ca_2+"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e3, index=("Liq", "Mg_2+"))
    m.fs.prop_od.set_default_scaling("flow_mass_phase_comp", 1e3, index=("Liq", "SO4_2-"))
   
    iscale.set_scaling_factor(desal.Elec.cell_width, 5)
    iscale.set_scaling_factor(desal.Elec.cell_length, 1)
    iscale.set_scaling_factor(desal.Elec.cell_pair_num, 0.1)
    iscale.set_scaling_factor(desal.Elec.voltage_applied, 1)
    iscale.set_scaling_factor(desal.P0.control_volume.work, 1e1)
    iscale.set_scaling_factor(desal.P1.control_volume.work, 1e1)
    iscale.set_scaling_factor(desal.ERD.control_volume.work, 1e-5)
    #H2 extraction
    #Oxygen Compression
    iscale.set_scaling_factor(H2x.Compressor_O2.feed_side.work, 1e-5)
    iscale.set_scaling_factor(H2x.Compressor_O2.brine_side.work, 1e-7)
    
    #Hydrogen Compression
    iscale.set_scaling_factor(H2x.Compressor_H2.feed_side.work, 1e-5)
    iscale.set_scaling_factor(H2x.Compressor_H2.brine_side.work, 1e-7)
    
    #Calculate and propagate scaling factors
    iscale.calculate_scaling_factors(m)

    return m

def set_operating_conditions(m):
    
    prtrt = m.fs.pretreatment = Block()
    Mgx = m.fs.Mgextraction = Block()
    Mgxstor = m.fs.Mgstorage = Block()
    Lix = m.fs.Liextraction = Block()
    Lixstor = m.fs.Listorage = Block()
    desal = m.fs.desalination = Block()
    psttrt = m.fs.posttreatment = Block()
    H2x = m.fs.H2extraction = Block()
    H2xstor = m.fs.H2storage = Block()

    # ---Specifications---
    #Feed

    flow_vol = 0.194 * pyunits.m**3 / pyunits.s
    conc_mass_Li = 1.7e-7 * pyunits.kg / pyunits.m**3
    conc_mass_Mg = 0.00128  * pyunits.kg / pyunits.m**3
    conc_mass_Na = 0.0108 * pyunits.kg / pyunits.m**3
    conc_mass_Ca = 0.0004 * pyunits.kg / pyunits.m**3
    conc_mass_Cl = 0.0194 * pyunits.kg / pyunits.m**3
    conc_mass_SO4 = 0.0027 * pyunits.kg / pyunits.m**3
    conc_mass_H = 0.06 * pyunits.kg / pyunits.m**3
    conc_mass_OH = 0.0000214 *pyunits.kg / pyunits.m**3
    conc_mass_H2 = 0.12 * pyunits.kg / pyunits.m**3
    conc_mass_O2 = 0.477 * pyunits.kg / pyunits.m**3
    conc_mass_tss = 0.03 * pyunits.kg / pyunits.m**3
    temperature = 293.15 * pyunits.K
    pressure = 101325 * pyunits.Pa 

    m.fs.feed.flow_vol[0].fix(flow_vol)
    m.fs.feed.conc_mass_comp[0, "Li_+"].fix(conc_mass_Li)
    m.fs.feed.conc_mass_comp[0, "Mg_2+"].fix(conc_mass_Mg)
    m.fs.feed.conc_mass_comp[0, "Na_+"].fix(conc_mass_Na)
    m.fs.feed.conc_mass_comp[0, "Ca_2+"].fix(conc_mass_Ca)
    m.fs.feed.conc_mass_comp[0, "Cl_-"].fix(conc_mass_Cl)
    m.fs.feed.conc_mass_comp[0, "SO4_2-"].fix(conc_mass_SO4)
    m.fs.feed.conc_mass_comp[0, "H_+"].fix(conc_mass_H)
    m.fs.feed.conc_mass_comp[0, "OH_-"].fix(conc_mass_OH)
    m.fs.feed.conc_mass_comp[0, "H2-v"].fix(conc_mass_H2)
    m.fs.feed.conc_mass_comp[0, "O2-v"].fix(conc_mass_O2)
    m.fs.feed.conc_mass_comp[0, "tss"].fix(conc_mass_tss)
    solve(m.fs.feed, checkpoint="solve feed block")

    m.fs.tb_prtrt_Mgx.properties_out[0].temperature.fix(temperature)
    m.fs.tb_prtrt_Mgx.properties_out[0].pressure.fix(pressure)

    #Pretreatment

    #Intake
    if m.x_location == "onshore":
        prtrt.intake.load_parameters_from_database()
    elif m.x_location == "offshore":
        m.db.get_unit_operation_parameters("raw")
        prtrt.intake.lift_height.fix(80)
        prtrt.intake.eta_pump.fix(0.9)
        prtrt.intake.eta_motor.fix(0.9)
        
    #Ferric chloride
    m.db.get_unit_operation_parameters("chemical_addition")
    prtrt.ferric_chloride_addition.load_parameters_from_database()
    prtrt.ferric_chloride_addition.chemical_dosage.fix(20)

    #Chlorination
    m.db.get_unit_operation_parameters("chlorination")
    prtrt.chlorination.load_parameters_from_database(use_default_removal=True)

    #Static mixer
    m.db.get_unit_operation_parameters("static_mixer")
    prtrt.static_mixer.load_parameters_from_database(use_default_removal=True)

    #Storage
    m.db.get_unit_operation_parameters("storage_tank")
    prtrt.storage.load_parameters_from_database(use_default_removal=True)
    prtrt.storage.storage_time.fix(2)

    #Screening
    m.db.get_unit_operation_parameters("screen")
    prtrt.screening.load_parameters_from_database(use_default_removal=True)

    #Coagulation and Flocculation
    m.db.get_unit_operation_parameters("coag_and_floc")
    prtrt.coag_and_floc.load_parameters_from_database(use_default_removal=True)

    #Sedimentation
    m.db.get_unit_operation_parameters("sedimentation")
    prtrt.sedimentation.load_parameters_from_database(use_default_removal=True)

    #Air Flotation
    m.db.get_unit_operation_parameters("air_flotation")
    prtrt.flotation.load_parameters_from_database(use_default_removal=True)

    #Gravity-Basin
    m.db.get_unit_operation_parameters("fixed_bed")
    prtrt.gravity_basin.load_parameters_from_database(use_default_removal=True)

    #Mediafiltration
    m.db.get_unit_operation_parameters("media_filtration")
    prtrt.mfiltration.load_parameters_from_database(use_default_removal=True)

    #Mediafiltration backwash
    m.db.get_unit_operation_parameters("backwash_solids_handling")
    prtrt.mbackwash_pump.load_parameters_from_database(use_default_removal=True)

    #Antiscalant addition
    prtrt.antiscalant_addition.load_parameters_from_database()
    prtrt.antiscalant_addition.chemical_dosage.fix(5)

    #Cartridge filtration
    m.db.get_unit_operation_parameters("cartridge_filtration")
    prtrt.cfiltration.load_parameters_from_database(use_default_removal=True)

    #GAC
    m.db.get_unit_operation_parameters("gac")
    prtrt.gac.load_parameters_from_database(use_default_removal=True)

    #GAC backwash
    m.db.get_unit_operation_parameters("backwash_solids_handling")
    prtrt.gbackwash_pump.load_parameters_from_database(use_default_removal=True)

    #Mediafiltration disposal
    m.db.get_unit_operation_parameters("landfill")
    prtrt.mf_disposal.load_parameters_from_database()

    #GAC filtration disposal
    m.db.get_unit_operation_parameters("landfill")
    prtrt.gac_disposal.load_parameters_from_database()

    #Powered Softening (Magnesium Nanofiltration)
    Mgx.P1.efficiency_pump.fix(0.80)
    nano_pressure = 172369 * pyunits.Pa #25 psi (20 psi is the minimum feed pressure for nano)
    Mgx.P1.control_volume.properties_out[0].pressure.fix(nano_pressure)

    Mgx.feed_side.properties_in[0].flow_mass_phase_comp

    Mgx.feed_side.properties_in[0].pressure.fix(172369)
    Mgx.feed_side.properties_in[0].temperature.fix(293.15)
    Mgx.flux_vol_solvent.fix(1.67e-6)
    Mgx.area.fix(500)
    Mgx.properties_permeate[0].pressure.fix(101325)

    Mgx.rejection_phase_comp[0, "Liq", "Li_+"].fix(0)
    Mgx.rejection_phase_comp[0, "Liq", "Na_+" ].fix(0.01)
    Mgx.rejection_phase_comp[0, "Liq", "Ca_2+"].fix(0.79)
    Mgx.rejection_phase_comp[0, "Liq", "Mg_2+"].fix(0.94)
    Mgx.rejection_phase_comp[0, "Liq", "SO4_2-"].fix(0.87)
    Mgx.rejection_phase_comp[0, "Liq", "Cl_-"].fix(0.15)
    Mgx.rejection_phase_comp[0, "Liq", "H_+"].fix(0)
    Mgx.rejection_phase_comp[0, "Liq", "OH_-"].fix(0)
    Mgx.rejection_phase_comp[0, "Liq", "O2-v"].fix(0)
    Mgx.rejection_phase_comp[0, "Liq", "H2-v"].fix(0)


    Mgx.feed_side.properties_in[0].assert_electroneutrality(
        defined_state=True, adjust_by_ion="Cl_-"
    )

    #Magnesium storage & Waste disposal
    Mgxstor.storage.load_parameters_from_database(use_default_removal=True)
    Mgxstor.storage.storage_time(1)
    Mgxstor.disposal.load_parameters_from_database(use_default_removal=True)
    Mgxstor.landfil.load_parameters_from_database(use_default_removal=True)

    #Li extraction
    Lix.P1.efficiency_pump.fix(0.80)
    operating_pressure = 101325 * pyunits.Pa
    Lix.P1.control_volume.properties_out[0].pressure.fix(operating_pressure)

    m.fs.prop_od.properties[0].pressure.fix(101325) 
    m.fs.prop_od.properties[0].temperature.fix(293.15)
    m.fs.prop_od.properties[0].flow_mol_phase_comp["Liq", "H2O"].fix(4.8)
    m.fs.prop_od.properties[0].flow_mol_phase_comp["Liq", "Li_+"].fix(1.476e-2)
    m.fs.prop_od.properties[0].flow_mol_phase_comp["Liq", "Cl_-"].fix(1.476e-2)

    #Fix separator's split fraction 
    Lix.S1.split_fraction[0, "inlet_diluate"].fix(0.5)

    #Fix ED unit vars
    Lix.Elec.water_trans_number_membrane["cem"].fix(5.8)
    Lix.Elec.water_trans_number_membrane["aem"].fix(4.3)
    Lix.Elec.water_permeability_membrane["cem"].fix(2.16e-14)
    Lix.Elec.water_permeability_membrane["aem"].fix(1.75e-14)
    Lix.Elec.voltage_applied.fix(5)
    Lix.Elec.electrodes_resistance.fix(0)
    Lix.Elec.cell_pair_num.fix(100)
    Lix.Elec.current_utilization.fix(1)
    Lix.Elec.channel_height.fix(2.7e-4)
    Lix.Elec.membrane_areal_resistance["cem"].fix(1.89e-4)
    Lix.Elec.membrane_areal_resistance["aem"].fix(1.77e-4)
    Lix.Elec.cell_width.fix(0.1)
    Lix.Elec.cell_length.fix(0.79)
    Lix.Elec.membrane_thickness["cem"].fix(1.3e-4)
    Lix.Elec.membrane_thickness["aem"].fix(1.3e-4)
    Lix.Elec.solute_diffusivity_membrane["cem", "Li_+"].fix(1.8e-10)
    Lix.Elec.solute_diffusivity_membrane["aem", "Li_+"].fix(1.25e-10)
    Lix.Elec.solute_diffusivity_membrane["cem", "Cl_-"].fix(1.8e-10)
    Lix.Elec.solute_diffusivity_membrane["aem", "Cl_-"].fix(1.25e-10)
    Lix.Elec.ion_trans_number_membrane["cem", "Li_+"].fix(1)
    Lix.Elec.ion_trans_number_membrane["aem", "Li_+"].fix(0)
    Lix.Elec.ion_trans_number_membrane["cem", "Cl_-"].fix(0)
    Lix.Elec.ion_trans_number_membrane["aem", "Cl_-"].fix(1)
    Lix.Elec.spacer_porosity.fix(1)

    #Lithium Storage & Waste disposal
    Lixstor.storage.load_parameters_from_database(use_default_removal=True)
    Lixstor.storage.storage_time(1)
    Lixstor.disposal.load_parameters_from_database(use_default_removal=True)
    Lixstor.landfill.load_parameters_from_database(use_default_removal=True)

    #Electrodialysis desalination
    m.fs.prop_od.properties[0].pressure.fix(101325) 
    m.fs.prop_od.properties[0].temperature.fix(293.15)
    desal.P0.control_volume.properties[0].pressure.fix(101325)
    desal.P1.efficiency_pump.fix(0.8)
    desal.P0.efficiency_pump.fix(0.8)

    #Fix separator's split fraction 
    desal.s0.split_fraction[0, "to_dil_in"].fix(0.5)
    desal.S1.split_fraction[0, "to_conc_in1"].fix(
        max(2-value(desal.recovery_vol_H2O) ** -1), 1e-8)
    desal.product.properties[0].pressure.fix(101325)
    desal.disposal.properties[0].pressure.fix(101325)
    desal.disposal.properties[0].temperature.fix(293.15)
    
    desal.Elec.water_trans_number_membrane["cem"].fix(5.8)
    desal.Elec.water_trans_number_membrane["aem"].fix(4.3)
    desal.Elec.water_permeability_membrane["cem"].fix(2.16e-14)
    desal.Elec.water_permeability_membrane["aem"].fix(1.75e-14)
    desal.Elec.voltage_applied.fix(5)
    desal.Elec.electrodes_resistance.fix(0)
    desal.Elec.cell_pair_num.fix(100)
    desal.Elec.current_utilization.fix(1)
    desal.Elec.channel_height.fix(2.7e-4)
    desal.Elec.membrane_areal_resistance["cem"].fix(1.89e-4)
    desal.Elec.membrane_areal_resistance["aem"].fix(1.77e-4)
    desal.Elec.cell_width.fix(0.1)
    desal.Elec.cell_length.fix(0.79)
    desal.Elec.membrane_thickness["cem"].fix(1.3e-4)
    desal.Elec.membrane_thickness["aem"].fix(1.3e-4)
    desal.Elec.solute_diffusivity_membrane["cem", "Na_+"].fix(1.8e-10)
    desal.Elec.solute_diffusivity_membrane["aem", "Na_+"].fix(1.25e-10)
    desal.Elec.solute_diffusivity_membrane["cem", "Cl_-"].fix(1.8e-10)
    desal.Elec.solute_diffusivity_membrane["aem", "Cl_-"].fix(1.25e-10)
    desal.Elec.solute_diffusivity_membrane["cem", "Mg_2+"].fix(1.8e-10)
    desal.Elec.solute_diffusivity_membrane["aem", "Mg_2+"].fix(1.25e-10)
    desal.Elec.solute_diffusivity_membrane["cem", "Ca_2+"].fix(1.8e-10)
    desal.Elec.solute_diffusivity_membrane["aem", "Ca_2+"].fix(1.25e-10)
    desal.Elec.solute_diffusivity_membrane["cem", "SO4_2-"].fix(1.8e-10)
    desal.Elec.solute_diffusivity_membrane["aem", "SO4_2-"].fix(1.25e-10)
    desal.Elec.ion_trans_number_membrane["cem", "Na_+"].fix(1)
    desal.Elec.ion_trans_number_membrane["aem", "Na_+"].fix(0)
    desal.Elec.ion_trans_number_membrane["cem", "Mg_2+"].fix(1)
    desal.Elec.ion_trans_number_membrane["aem", "Mg_2+"].fix(0)
    desal.Elec.ion_trans_number_membrane["cem", "Ca_2+"].fix(1)
    desal.Elec.ion_trans_number_membrane["aem", "Ca_2+"].fix(0)
    desal.Elec.ion_trans_number_membrane["cem", "Cl_-"].fix(0)
    desal.Elec.ion_trans_number_membrane["aem", "Cl_-"].fix(1)
    desal.Elec.ion_trans_number_membrane["cem", "SO4_2-"].fix(0)
    desal.Elec.ion_trans_number_membrane["aem", "SO4_2-"].fix(1)

    # Stack properties
    desal.Elec.cell_pair_num.fix(56)
    desal.Elec.channel_height.fix(7.1e-4)
    desal.Elec.cell_width.fix(0.197)
    desal.Elec.cell_length.fix(1.68)

    # Spacer properties
    desal.Elec.spacer_porosity.fix(0.83)
    desal.Elec.spacer_specific_area.fix(10400)

    # Electrochemical properties
    desal.Elec.electrodes_resistance.fix(0)
    desal.Elec.current_utilization.fix(1)
    desal.Elec.diffus_mass.fix(1.6e-9)

    desal.ERD.efficiency_pump.fix(0.95)
    desal.ERD.control_volume.properties_out[0].pressure.fix(101325)

    #Desalination posttreatment

    #Demineralization
    m.db.get_unit_operation_parameters("ion_exchange")
    psttrt.IX.load_parameters_from_database(use_default_removal=True)
    #update resin replacement rate based on AmberLite™ MB20 H/OH Ion Exchange Resin

    #Product water storage
    m.db.get_unit_operation_parameters("storage_tank")
    psttrt.storage.load_parameters_from_database(use_default_removal=True)
    psttrt.storage.storage_time.fix(2)
    psttrt.disposal.load_parameters_from_database(use_default_removal=True)
    psttrt.landfill.load_parameters_from_database(use_default_removal=True)

    #H2 Extraction
    #Water Heating
    H2x.Heat.heat_duty[0].fix(125e6)
    #Fix separator's split fraction 
    H2x.S1.split_fraction[0, "inlet_cathode"].fix(0.5)
    if m.elec_type == "PEM":
        m.fs.tb_psttrt_H2x.properties_out[0].temperature.fix(273.15 + 70)
        m.fs.tb_psttrt_H2x.properties_out[0].pressure.fix(30e5)

        #Anolyte block
        H2x.ER.anolyte.properties_in[0].temperature.fix(273.15 + 70)
        H2x.ER.anolyte.properties_in[0].pressure.fix(30e5)
        H2x.ER.anolyte.properties_in[0].flow_mol_phase_comp["Liq", "H2O"].fix(5.551)
        H2x.ER.anolyte.properties_in[0].flow_mol_phase_comp["Liq", "H_+"].fix(0)
        H2x.ER.anolyte.properties_in[0].flow_mol_phase_comp["Liq", "O2-v"].fix(0)
        H2x.ER.anolyte.properties_in[0].flow_mol_phase_comp["Liq", "H2-v"].fix(0)

        #Catholyte block
        H2x.ER.catholyte.properties_in[0].temperature.fix(273.15 + 70)
        H2x.ER.catholyte.properties_in[0].pressure.fix(30e5)
        H2x.ER.catholyte.properties_in[0].flow_mol_phase_comp["Liq", "H2O"].fix(5.551)
        H2x.ER.catholyte.properties_in[0].flow_mol_phase_comp["Liq", "H_+"].fix(0)
        H2x.ER.catholyte.properties_in[0].flow_mol_phase_comp["Liq", "O2-v"].fix(0)
        H2x.ER.catholyte.properties_in[0].flow_mol_phase_comp["Liq", "H2-v"].fix(0)

        #PEM reactions
        H2x.ER.membrane_ion_transport_number["Liq", "H_+"].fix(1)
        # H2O --> 0.5 O2 + 2H+ +2e (oxidation reaction)
        H2x.ER.anode_electrochem_potential.fix(1.23)
        H2x.ER.anode_stoich["Liq", "H2O"].fix(-1)
        H2x.ER.anode_stoich["Liq", "O2-v"].fix(0.5) #gaseous 
        H2x.ER.anode_stoich["Liq", "H_+"].fix(2)

        # cathode properties 
        # 2H+ + 2e- --> H2 (reduction reaction)
        H2x.ER.cathode_electrochem_potential.fix(0.000)
        H2x.ER.cathode_stoich["Liq", "H_+"].fix(-2)
        H2x.ER.cathode_stoich["Liq", "H2-v"].fix(1) #gaseous

        #PEM Electrolysis unit properties: 
        H2x.ER.membrane_current_density.fix(3e4) #membrane current density (A/m^2)
        H2x.ER.anode_current_density.fix(3e4) #anode current density (A/m^2) 
        H2x.ER.anode_overpotential.fix(3e-2) #anode overpotential (V)  source:https://www.sciencedirect.com/science/article/pii/S0378775311018131
        H2x.ER.cathode_current_density.fix(3e4) #cathode current density (A/m^2) 
        H2x.ER.cathode_overpotential.fix(3e-2) #cathode overpotential (V) source:https://www.sciencedirect.com/science/article/pii/S0378775311018131
        H2x.ER.current.fix(1.875e4) #current (A) source:https://www.sciencedirect.com/science/article/pii/S0378775311018131
        H2x.ER.efficiency_current.fix(0.65) #current efficiency (-) source:https://onlinelibrary.wiley.com/doi/epdf/10.1002/sstr.202200130
        H2x.ER.resistance.fix(1.1e-4) #ohmic resistance (mΩ cm2,unit discrepancy) source: https://iopscience.iop.org/article/10.1149/08613.0695ecst/pdf
        
    elif m.elec_type == "AWE":
        m.fs.tb_psttrt_H2x.properties_out[0].temperature.fix(273.15 + 70)
        m.fs.tb_psttrt_H2x.properties_out[0].pressure.fix(30e5)
        
        #Anolyte Block
        H2x.ER.anolyte.properties_in[0].temperature.fix(273.15 + 70)
        H2x.ER.anolyte.properties_in[0].pressure.fix(30e5)
        H2x.ER.anolyte.properties_in[0].flow_mass_phase_comp["Liq", "H2O"].fix(5.551)
        H2x.ER.anolyte.properties_in[0].flow_mass_phase_comp["Liq", "OH_-"].fix(0)
        H2x.ER.anolyte.properties_in[0].flow_mass_phase_comp["Liq", "O2-v"].fix(2.22)
        H2x.ER.anolyte.properties_in[0].flow_mass_phase_comp["Liq", "H2-v"].fix(0) 

        #Catholyte Block
        H2x.ER.catholyte.properties_in[0].temperature.fix(273.15 + 70)
        H2x.ER.catholyte.properties_in[0].pressure.fix(30e5)
        H2x.ER.catholyte.properties_in[0].flow_mass_phase_comp["Liq", "H2O"].fix(5.551)
        H2x.ER.catholyte.properties_in[0].flow_mass_phase_comp["Liq", "OH_-"].fix(1.5)
        H2x.ER.catholyte.properties_in[0].flow_mass_phase_comp["Liq", "O2-v"].fix(0)
        H2x.ER.catholyte.properties_in[0].flow_mass_phase_comp["Liq", "H2-v"].fix(4.44) 
    
        #Alkaline Water Electrolysis reactions 
        #Assumes 30% weight KOH electrolyte introduced to the anode and cathode
        H2x.ER.dens_mass_const.fix(1063)
        H2x.ER.membrane_ion_transport_number["Liq", "OH_-"].fix(1)

        #anode properties
        # 4OH- --> 2H2O + O2 + 4e- 
        H2x.ER.anode_electrochem_potential.fix(-0.4)
        H2x.ER.anode_stoich["Liq", "OH_-"].fix(-4)
        H2x.ER.anode_stoich["Liq", "H2O"].fix(2)
        H2x.ER.anode_stoich["Liq", "O2-v"].fix(1)

        #cathode properties
        # 4H2O + 4e- --> 2H2 + 4OH-
        H2x.ER.cathode_electrochem_potential.fix(-0.83)
        H2x.ER.cathode_stoich["Liq", "H2O"].fix(-4)
        H2x.ER.cathode_stoich["Liq", "H2-v"].fix(2)
        H2x.ER.cathode_stoich["Liq", "OH_-"].fix(4)

    #Electrolysis unit properties source: https://onlinelibrary.wiley.com/doi/epdf/10.1002/sstr.202200130
    H2x.ER.custom_reaction_anode[0, "O2-v"].fix()
    H2x.ER.custom_reaction_cathode[0, "H2-v"].fix()
    H2x.ER.membrane_current_density.fix(3e3) #membrane current density (A/m^2)
    H2x.ER.anode_current_density.fix(3e3) #anode current density (A/m^2) 
    H2x.ER.anode_overpotential.fix(9.39e-1) #anode overpotential (V)  source:https://www.sciencedirect.com/science/article/pii/S0378775311018131
    H2x.ER.cathode_current_density.fix(3e3) #cathode current density (A/m^2) 
    H2x.ER.cathode_overpotential.fix(9.6e-1) #cathode overpotential (V) source:https://www.sciencedirect.com/science/article/pii/S0378775311018131
    H2x.ER.current.fix(2.25e4) #current (A) source:https://www.sciencedirect.com/science/article/pii/S0378775311018131
    H2x.ER.efficiency_current.fix(0.61) #current efficiency (-) source:https://onlinelibrary.wiley.com/doi/epdf/10.1002/sstr.202200130
    H2x.ER.resistance.fix(8.18e-5) #ohmic resistance (mΩ cm2,unit discrepancy) source: https://iopscience.iop.org/article/10.1149/08613.0695ecst/pdf

    #Produced gas compression
    H2x.Compressor_H2.ratioP.fix(35e6)
    H2x.Compressor_O2.ratioP.fix(14e6)
    H2x.Compressor_O2.efficiency_pump.fix(0.95)
    H2x.Compressor_H2.efficiency_pump.fix(0.95)
    #Produced gas storage
    H2xstor.O2.load_parameters_from_database(use_default_removal=True)
    H2xstor.O2.storage_time.fix(24)

    H2xstor.H2.load_parameters_from_database(use_default_removal=True)
    H2xstor.H2.storage_time(24)

def initialize_system(m):
        prtrt = m.fs.pretreatment 
        Mgx = m.fs.Mgextraction
        Mgxstor = m.fs.Mgstorage
        Lix = m.fs.Liextraction 
        Lixstor =m.fs.Listorage 
        desal = m.fs.desalination 
        psttrt = m.fs.posttreatment
        H2x = m.fs.H2extraction 
        H2xstor = m.fs.H2storage 

        if solver is None: 
            solver = get_solver()
        optarg= solver.options 

        #initialize feed
        solve(m.fs.feed, checkpoint="solve flowsheet after initialize feed")

        #initialize pretreatment
        propagate_state(m.fs.s_feed)
        flags = fix_state_vars(prtrt.intake.properties)
        solve(prtrt, checkpoint="solve flowsheet after initializing pre-treatment")
        revert_state_vars(prtrt.intake.properties, flags)

        #initialize Li extraction
        propagate_state(m.fs.s_prtrt_tb)
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "H2O"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["H2O"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "Li_+"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["Li_+"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "Ca_2+"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["Ca_2+"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "Mg_2+"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["Mg_2+"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "Cl_-"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["Cl_-"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "SO4_2-"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["SO4_2-"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "Na_+"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["Na_+"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "H_+"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["H_+"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "OH_-"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["OH_-"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "O2-v"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["O2-v"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "H2-v"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["H2-v"]
        )
        m.fs.tb_prtrt_Mgx.properties_out[0].flow_mass_phase_comp["Liq", "tss"] = value(
            m.fs.tb_prtrt_Mgx.properties_in[0].flow_mass_comp["tss"]
        )
        Mgx.P1.initialize()
        Mgx.S1.initialize()
        Mgx.nano.initialize()

        Lix.P1.initialize()
        Lix.IX.initialize()
        propagate_state(m.fs.s_tb_Lix)
        
        #initialize Lithium storage 
        propagate_state(m.fs.s_Lix_tb)
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["H2O"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "H2O"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["Li_+"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "Li_+"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["Ca_2+"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "Ca_2+"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["Mg_2+"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "Mg_2+"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["Cl_-"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "Cl_-"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["SO4_2-"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "SO4_2-"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["Na_+"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "Na_+"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["H_+"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "H_+"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["OH_-"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "OH_-"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["O2-v"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "O2-v"]
        )
        m.fs.tb_Lix_Lixstor.properties_out[0].flow_mass_comp["H2-v"] = value(
            m.fs.tb_Lix_Lixstor.properties_in[0].flow_mass_phase_comp["Liq", "H2-v"]
        )
        propagate_state(m.fs.s_tb_Lixstor)
        flags = fix_state_vars(Lixstor.storage.properties)
        solve(Lixstor, checkpoint="solve flowsheet after initializing Lithium Storage")
        revert_state_vars(Lixstor.storage.properties, flags)

        propagate_state(m.fs.s_Lixstor_tb)
        #Translator Block
        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "H2O"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["H2O"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "Li_+"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["Li_+"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "Ca_2+"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["Ca_2+"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "Mg_2+"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["Mg_2+"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "Cl_-"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["Cl_-"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "SO4_2-"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["SO4_2-"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "Na_+"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["Na_+"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "H_+"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["H_+"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "OH_-"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["OH_-"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "O2-v"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["O2-v"]
        )

        m.fs.tb_Lixstor_desal.properties_out[0].flow_mass_phase_comp["Liq", "H2-v"] = value(
            m.fs.tb_Lixstor_desal.properties_in[0].flow_mass_comp["H2-v"]
        )
        #initalize electrodialysis Desalination
        propagate_state(desal.s01)
        desal.S0.initialize(optarg=optarg)
        propagate_state(desal.s02)
        desal.P1.deltaP[0].fix(2e5)
        desal.P1.initialize()
        desal.P1.deltaP[0].unfix()
        propagate_state()
        #initialize posttreatment
        propagate_state(m.fs.s_desal_tb)
        m.fs.tb_desal_psttrt.properties_out[0].flow_mass_comp["H2O"] = value(
            m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "H2O"]
        )
        m.fs.tb_desal_psttrt.properties_out[0].flow_mass_comp["tds"] = value(
            m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "Li_+"]
            + value(m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "Ca_2+"])
            + value(m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "Mg_2+"])
            + value(m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "Cl_-"])
            + value(m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "SO4_2-"])
            + value(m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "Na_+"])
        )
        m.fs.tb_desal_psttrt.properties_out[0].flow_mass_comp["H_+"] = value(
            m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "H_+"]
        )
        m.fs.tb_desal_psttrt.properties_out[0].flow_mass_comp["OH_-"] = value(
            m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "OH_-"]
        )
        m.fs.tb_desal_psttrt.properties_out[0].flow_mass_comp["O2-v"] = value(
            m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "O2-v"]
        )
        m.fs.tb_desal_psttrt.properties_out[0].flow_mass_comp["H2-v"] = value(
            m.fs.tb_desal_psttrt.properties_in[0].flow_mass_phase_comp["Liq", "H2-v"]
        )
        propagate_state(m.fs.s_tb_psttrt)
        flags = fix_state_vars(psttrt.IX.properties)
        solve(psttrt, checkpoint="solve flowsheet after initializing post-treatment")
        revert_state_vars(psttrt.IX.properties, flags)


        #initialize H2 extraction
        propagate_state(m.fs.s_psttrt_tb)
        m.fs.tb_psttrt_H2x.properties_out[0].flow_mass_phase_comp["Liq", "H2O"] = value(
            m.fs.tb_psttrt_H2x.properties_in[0].flow_mass_comp["H2O"]
        )
        m.fs.tb_psttrt_H2x.properties_out[0].flow_mass_phase_comp["Liq", "tds"] = value(
            m.fs.tb_psttrt_H2x.properties_in[0].flow_mass_comp["tds"]
        )
        m.fs.tb_psttrt_H2x.properties_out[0].flow_mass_phase_comp["Liq", "H_+"] = value(
            m.fs.tb_psttrt_H2x.properties_in[0].flow_mass_comp["H_+"]
        )
        m.fs.tb_psttrt_H2x.properties_out[0].flow_mass_phase_comp["Liq", "OH_-"] = value(
            m.fs.
            tb_psttrt_H2x.properties_in[0].flow_mass_comp["OH_-"]
        )
        m.fs.tb_psttrt_H2x.properties_out[0].flow_mass_phase_comp["Liq", "O2-v"] = value(
            m.fs.tb_psttrt_H2x.properties_in[0].flow_mass_comp["O2-v"]
        )
        m.fs.tb_psttrt_H2x.properties_out[0].flow_mass_phase_comp["Liq", "H2-v"] = value(
            m.fs.tb_psttrt_H2x.properties_in[0].flow_mass_comp["H2-v"]
        )
        H2x.ER.initialize()
        if m.elec_type == "PEM":
            solve(
                H2x,
                checkpoint=f"solve flowsheet after initializing {m.elec_type} electrolysis"
            )
        elif m.elec_type == "AWE":
            solve(
                H2x,
                checkpoint=f"solve flowsheet after initializing {m.elec_type} electrolysis"
            )
        propagate_state(m.fs.s_tb_H2)
        propagate_state(m.fs.s_tb_O2)

        H2x.Compressor_H2.initialize()
        H2x.Compressor_O2.initialize()

        propagate_state(m.fs.s_H2_tb)
        propagate_state(m.fs.s_O2_tb)
        m.fs.tb_H2x_H2xstor.properties_out[0].flow_mass_comp["H2-v"] = value(
            m.fs.tb_H2x_H2xstor.properties_in[0].flow_mass_phase_comp["Liq", "H2-v"]
        )
        m.fs.tb_H2x_H2xstor.properties_out[0].flow_mass_comp["O2-v"] = value(
            m.fs.tb_H2x_H2xstor.properties_in[0].flow_mass_phase_comp["Liq", "O2-v"]
        )
        propagate_state(m.fs.s_tb_H2xstor_H2)
        propagate_state(m.fs.s_tb_H2xstor_O2)

        flags = fix_state_vars(H2xstor.H2.properties)
        solve(H2xstor, checkpoint="solve flowsheet after initializing gas storage")
        revert_state_vars(H2xstor.H2.properties, flags)
        

def optimize_operation(m): 
    Mgx = m.fs.Mgextraction
    Lix = m.fs.Liextraction  
    desal = m.fs.desalination 
    H2x = m.fs.H2extraction

    #Mg extraction unit
    m.fs.prop_od.properties[0].flow_vol_phase["Liq"].fix()
    m.fs.prop_od.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"].fix()
    m.fs.prop_od.properties[0].flow_mol_phase_comp["Liq", "Mg_2+"].unfix()
    m.fs.prop_od.properties[0].conc_mass_phase_comp["Liq", "H2O"].unfix()
    m.fs.prop_od.properties[0].flow_mol_phase_comp["Liq", "H2O"].unfix()

    Mgx.P1.control_volume.properties_out[0].pressure.unfix()
    Mgx.P1.control_volume.properties_out[0].pressure.setub(8300000) #pressure vessel burst pressure
    Mgx.P1.control_volume.properties_out[0].pressure.setlb(100000)
    
    Mgx.nano.area.unfix()
    Mgx.nano.velocity.unfix()
    Mgx.nano.velocity.setub(0.25)
    Mgx.treated.max_hardness.fix(100)
    Mgx.nano.recovery_vol_phase[0.0, "Liq"].setub(0.9)
    m.fs.prop_od.properties[0].total_hardness
    Mgx.byproduct.properties[0].total_hardness

    

    #Li extraction unit 
    Lix.P1.control_volume.properties_out[0].pressure.unfix()
    Lix.P1.control_volume.properties_out[0].pressure.setub(8300000) #pressure vessel burst pressure
    Lix.P1.control_volume.properties_out[0].pressure.setlb(100000)

    #Choose and unfix variables to be optimized
    Lix.Elec.voltage_applied[0].unfix()
    Lix.Elec.cell_pair_num.unfix()
    Lix.Elec.cell_pair_num.set_value(10)

    #Give narrower bounds to optimizing variables if available
    Lix.Elec.voltage_applied[0].setlb(0.5)
    Lix.Elec.voltage_applied[0].setub(20)
    Lix.Elec.cell_pair_num.setlb(5)
    Lix.Elec.cell_pair_num.setub(500)

    #Electrodialysis desalination
    ulim = (
    desal.Elec.voltage_x[0,0].value
    / desal.Elec.current_density_x[0,0].value
    * desal.Elec.current_dens_lim_x[0, 0].value
    * 1.5
    )
    desal.Elec.voltage_applied[0].unfix()
    desal.Elec.voltage_applied[0].setlb(0.1)
    desal.Elec.voltage_applied[0].setub(ulim)
    desal.Elec.cell_pair_num.unfix()
    desal.Elec.cell_pair_num.setlb(10)
    desal.Elec.cell_pair_num.setub(1000)
    desal.Elec.cell_length.unfix()
    desal.product.properties[0].conc_mol_phase_comp["Liq", "Na_+"].fix(0.428) #Corresponding to product concentration of 25 ppm

    #Hydrogen Electrolysis
    if m.elec_type == "PEM":
    
        #Source: Zhang, Fan, et al. “The Survey of Key Technologies in Hydrogen Energy Storage.” 
        #International Journal of Hydrogen Energy, vol. 41, no. 33, Sept. 2016, pp. 14535–14552,
        # https://doi.org/10.1016/j.ijhydene.2016.05.293.
        
        H2x.ER.membrane_area.unfix()
        H2x.ER.membrane_area.setub(300)
        H2x.ER.membrane_area.setlb(0.5)

        H2x.ER.membrane_current_density.unfix()
        H2x.ER.membrane_current_density.setub(2)
        H2x.ER.membrane_current_density.setlb(0.6)

        H2x.ER.voltage_cell.unfix()
        H2x.ER.voltage_cell.setub(2.2)
        H2x.ER.voltage_cell.setlb(1.8)

        H2x.ER.voltage_efficiency.unfix()
        H2x.ER.voltage_efficiency.setub(0.82)
        H2x.ER.voltage_efficiency.setlb(0.67)

    elif m.elec_type == "AWE":

        #Source: Zhang, Fan, et al. “The Survey of Key Technologies in Hydrogen Energy Storage.” 
        #International Journal of Hydrogen Energy, vol. 41, no. 33, Sept. 2016, pp. 14535–14552,
        # https://doi.org/10.1016/j.ijhydene.2016.05.293.

        H2x.ER.membrane_area.unfix()
        H2x.ER.membrane_area.setub(4)
        H2x.ER.membrane_area.setlb(0.5)

        H2x.ER.membrane_current_density.unfix()
        H2x.ER.membrane_current_density.setub(0.4)
        H2x.ER.membrane_current_density.setlb(0.2)

        H2x.ER.voltage_cell.unfix()
        H2x.ER.voltage_cell.setub(2.4)
        H2x.ER.voltage_cell.setlb(1.8)

        H2x.ER.voltage_efficiency.unfix()
        H2x.ER.voltage_efficiency.setub(0.82)
        H2x.ER.voltage_efficiency.setlb(0.62)

        m.fs.objective = Objective(expr=m.fs.LCOT_MgLiH2O2)
        return


def solve(blk, solver=None, checkpoint=None, tee=False, fail_flag=True):
        if solver is None:
            solver = get_solver()
        results = solver.solve(blk, tee=tee)
        check_solve(results, checkpoint=checkpoint, logger=_log, fail_flag=fail_flag)
        return results 
    
def display_results(m):
        m.fs.feed.report()
        m.fs.pretreatment.intake.report()
        m.fs.pretreatment.ferric_chloride_addition.report()
        m.fs.pretreatment.chlorination.report()
        m.fs.pretreatment.static_mixer.report()
        m.fs.pretreatment.storage.report()
        m.fs.pretreatment.screening.report()
        m.fs.pretreatment.coag_and_floc.report()
        m.fs.pretreatment.sedimentation.report()
        m.fs.pretreatment.flotation.report()
        m.fs.pretreatment.gravity_basin.report()
        m.fs.pretreatment.mfiltration.report()
        m.fs.pretreatment.mbackwash_pump.report()
        m.fs.pretreatment.antiscalant_addition.report()
        m.fs.pretreatment.cfiltration.report()
        m.fs.pretreatment.gac.report()
        m.fs.pretreatment.gbackwash_pump.report()
        m.fs.pretreatment.mf_disposal.report()
        m.fs.pretreatment.gac_disposal.report
        m.fs.Mgextraction.P1.report()
        m.fs.Mgextraction.S1.report()
        m.fs.Mgextraction.nano.report()
        m.fs.Mgextraction.treated.report()
        m.fs.Mgextraction.byproduct.report()
        m.fs.Mgstorage.storage.report()
        m.fs.Mgstorage.disposal.report()
        m.fs.Mgstorage.landfill.report()
        m.fs.Liextraction.P1.report()
        m.fs.Liextraction.S1.report()
        m.fs.Liextraction.Elec.report()
        m.fs.Liextraction.product.report()
        m.fs.Liextraction.treated.report()
        m.fs.Listorage.storage.report()
        m.fs.Listorage.disposal.report()
        m.fs.Listorage.landfill.report()
        m.fs.desalination.S0.report()
        m.fs.desalination.M0.report()
        m.fs.desalination.P0.report()
        m.fs.desalination.P1.report()
        m.fs.desalination.Elec.report()
        m.fs.desalination.S1.report()
        m.fs.desalination.ERD.report()
        m.fs.desalination.product.report()
        m.fs.desalination.disposal.report()
        m.fs.posttreatment.IX.report()
        m.fs.posttreatment.storage.report()
        m.fs.posttreatment.disposal.report()
        m.fs.posttreatment.landfill.report()
        if m.elec_type == "PEM":
            m.fs.H2extraction.ER.report()
            m.fs.H2extraction.Compressor_O2.report()
            m.fs.H2extraction.Compressor_H2.report()
            m.fs.H2extraction.prod_O2.report()
            m.fs.H2extraction.prod_H2.report()
        elif m.elec_type == "AWE":
            m.fs.H2extraction.ER.report()
            m.fs.H2extraction.Compressor_O2.report()
            m.fs.H2extraction.Compressor_H2.report()
            m.fs.H2extraction.prod_O2.report()
            m.fs.H2extraction.prod_H2.report()
        m.fs.H2storage.O2.report()
        m.fs.H2storage.H2.report()

def add_costing(m):
    prtrt = m.fs.pretreatment 
    Mgx = m.fs.Mgextraction
    Mgxstor = m.fs.Mgstorage
    Lix = m.fs.Liextraction 
    Lixstor =m.fs.Listorage 
    desal = m.fs.desalination 
    psttrt = m.fs.posttreatment
    H2x = m.fs.H2extraction 
    H2xstor = m.fs.H2storage 

    
    #Add costing package for zero-order units
    m.fs.zo_costing = ZeroOrderCosting()
    m.fs.od_costing = WaterTAPCosting()

    #Add costing to zero order units 
    prtrt.intake.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.ferric_chloride_addition = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.chlorination = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.static_mixer = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.storage = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.screening = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.coag_and_floc = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.sedimentation = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.flotation = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.gravity_basin = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.mfiltration = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.mbackwash_pump = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.antiscalant_addition = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.cfiltration = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.gac = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.gbackwash_pump = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.mf_disposal = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    prtrt.gac_disposal = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

    #Magnesium extraction unit
    Mgx.P1.costing = UnitModelCostingBlock(
        flowsheet_costing_blolck=m.fs.od_costing,
        costing_method_arguments={"cost_electricity_flow": False},
    )
    Mgx.S1.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    Mgx.nano.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)

    #Magnesium storage & Waste disposal
    Mgxstor.storage.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    Mgxstor.disposal.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    Mgxstor.landfill.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

   #Li extraction unit 
    Lix.P1.costing = UnitModelCostingBlock(
        flowsheet_costing_block=m.fs.od_costing,
        costing_method_arguments={"cost_electricity_flow": False},
    )
    Lix.S1.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    Lix.Elec.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)

    #Lithium storage & Waste disposal
    Lixstor.storage.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    Lixstor.disposal.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    Lixstor.landfill.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

    #Electrodialysis Desalination
    desal.S0.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    desal.M0.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    desal.P0.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    desal.P1.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    desal.Elec.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    desal.S1.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
    m.fs.zo_costing.cost_flow(desal.P0.work_mechanical[0], "electricity")
    m.fs.zo_costing.cost_flow(desal.P1.work_mechanical[0], "electricity")
    m.fs.zo_costing.cost_flow(desal.FP_ERD.work_mechanical[0], "electricity")

    #Postreatment 
    psttrt.IX.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    psttrt.storage.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    psttrt.disposal.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    psttrt.landfill.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

    if m.elec_type == "PEM":
        H2x.ER.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
        #Source:https://www.sciencedirect.com/science/article/pii/S0360319923022590?ref=pdf_download&fr=RR-2&rr=905075ba29f8e818
        H2x.ER.costing.factor_membrane_replacement.fix(0.22)
        H2x.ER.costing.membrane_unit_cost.fix(2000)
        #Source: https://thundersaidenergy.com/downloads/nafion-membranes-costs-and-hydrogen-crossover/
        H2x.ER.costing.anode_unit_cost.fix(630)
        H2x.ER.costing.cathode_unit_cost.fix(630)
        H2x.Compressor_O2 = UnitModelCostingBlock(
            flowsheet_costing_block=m.fs.od_costing,
            costing_method_arguments={"cost_electricity_flow": True},
        )
        H2x.Compressor_H2 = UnitModelCostingBlock(
            flowsheet_costing_block=m.fs.od_costing,
            costing_method_arguments={"cost_electricity_flow": True},
        )
    if m.elec_type == "AWE":
        H2x.ER.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.od_costing)
        #Source:https://www.sciencedirect.com/science/article/pii/S0360319923022590?ref=pdf_download&fr=RR-2&rr=905075ba29f8e818
        H2x.ER.costing.factor_membrane_replacement.fix(0.33)
        H2x.ER.costing.membrane_unit_cost.fix(155)
        H2x.ER.costing.anode_unit_cost.fix(72)
        H2x.ER.costing.cathode_unit_cost.fix(72)
        H2x.Compressor_O2 = UnitModelCostingBlock(
            flowsheet_costing_block=m.fs.od_costing,
            costing_method_arguments={"cost_electricity_flow": True},
        )
        H2x.Compressor_H2 = UnitModelCostingBlock(
            flowsheet_costing_block=m.fs.od_costing,
            costing_method_arguments={"cost_electricity_flow": True},
        )
    H2xstor.O2.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
    H2xstor.H2.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

    #Aggregate unit level costs and calculate overall process costs
    m.fs.zo_costing.cost_process()
    
    
    feed_flowrate = m.fs.feed.flow_vol[0]
    m.fs.zo_costing.add_electricity_intensity(feed_flowrate)
    m.fs.od_costing.add_specific_energy_consumption(feed_flowrate)
    m.fs.od_costing.electricity_cost = value(m.fs.zo_costing.electricity_cost)
    m.fs.od_costing.base_currency = pyunits.USD_2020
    m.fs.od_costing.utilization_factor.fix(0.85) 
    zo_crf = m.fs.zo_costing.capital_recovery_factor
    m.fs.od_costing.capital_recovery_factor.fix(value(zo_crf))
    m.fs.od_costing.wacc.unfix()

    m.fs.od_costing.cost_process()

    m.fs.specific_energy_intensity = Expression(
        expr=(
            m.fs.zo_costing.electricity_intensity
            + m.fs.od_costing.specific_energy_consumption
        ),
        doc="Specific energy consumption of the treatment configuration on a feed flowrate basis [kWh/m3] "
    )

    #Annual disposal of waste
    m.fs.brine_disposal_cost = Expression(
        expr=(
            m.fs.zo_costing.utilization_factor
            * (
                m.fs.zo_costing.waste_disposal.cost
                * pyunits.convert(
                    desal.disposal.properties[0].flow_vol,
                    to_units=pyunits.m**3 /m.fs.zo_costing.base_period,


                )
            )
        ),
        doc="Cost of disposing of brine waste",
    )
    #Annual disposal of sludge 
    m.fs.sludge_disposal_cost = Expression(
        expr=(
            m.fs.zo_costing.utilization_factor
            * (
                m.fs.zo_costing.waste_disposal_cost
                * pyunits.convert(
                    prtrt.mf_disposal.properties[0].flow_vol
                    + prtrt.gac_disposal.properties[0].flow_vol,
                    + Mgxstor.landfill.properties[0].flow_vol,
                    + Lixstor.landfill.properties[0].flow_vol,
                    + psttrt.landfill.proeprties[0].flow_vol,
                    to_units=pyunits.m**3 / m.fs.zo_costing.base_period,

                )
            )
        ),
        doc="Cost of disposing pretreatment sludge"
    )
    #Annual water recovery
    m.fs.water_recovery_revenue = Expression(
        expr=(
            m.fs.zo_costing.utilization_factor
            * m.fs.zo_costing.recovered_water_cost
            * pyunits.convert(
                prtrt.gac.treated.properties[0].flow_vol
                + Mgx.nano.treated.properties[0].flow_vol,
                + Lix.Elec.product.properties[0].flow_vol,
                + desal.product.properties[0].flow_vol,
                + psttrt.IX.outlet.properties[0].flow_vol,
                to_units=pyunits.m**3 / m.fs.zo_costing.base_period,
            )
        ),
        doc="Savings from water recovered",
    )
    #Annual magnesium recovery
    m.fs.Mg_recovery_revenue = Expression(
        expr=(
            m.fs.zo_costing.utilization_factor
            * m.fs.zo_costing.recovered_Mg_cost
            *pyunits.convert(
                (Mgx.nano.inlet.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                - Mgx.nano.outlet.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"])
                * Mgx.nano.outlet.properties[0].flow_vol,
                pyunits.m**3 / m.fs.zo_costing.base_period,

            )
        ),
        doc="Savings from magnesium recovered"
    )
    #Annual lithium recovery
    m.fs.Li_recovery_revenue = Expression(
        expr=(
            m.fs.zo_costing.utilization_factor
            * m.fs.zo_costing.recovered_Li_cost
            *pyunits.convert(
                (Lix.Elec.inlet_concentrate.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                - Lix.Elec.outlet_concentrate.properties[0].conc_mass_phase_comp["Liq", "Li_+"])
                * Lix.Elec.outlet_concentrate.properties[0].flow_vol,
                pyunits.m**3 / m.fs.zo_costing.base_period,

            )
        ),
        doc="Savings from lithium recovered"
    )
    m.fs.H2_recovery_revenue = Expression(
        expr=(
            m.fs.zo_costing.utilization_factor
            * m.fs.zo_costing.recovered_H2_cost
            * pyunits.convert(
                H2x.prod_H2.inlet.properties[0].flow_vol,
                to_units=pyunits.m**3 / m.fs.zo_costing.base_period,
            )
        ),
        doc="Saving from hydrogen recovered"
    )
    m.fs.O2_recovery_revenue = Expression(
        expr=(
            m.fs.zo_costing.utilization_factor
            * m.fs.zo_costing.recovered_O2_cost
            * pyunits.convert(
                H2x.prod_O2.inlet.properties[0].flow_vol,
                to_units=pyunits.m**3 / m.fs.zo_costing.base_period
            )
        )
    )

    @m.fs.Expression(doc="Total capital cost of the treatment train")
    def total_capital_cost(b):
        return (
            pyunits.convert(
                m.fs.zo_costing.total_capital_cost, to_units=pyunits.USD_2020
            ) + pyunits.convert(
                m.fs.od_costing.total_capital_cost, to_units=pyunits.USD_2020
            )
        )
    
    @m.fs.Expression(doc="Total operating cost of the treatment train")
    def total_operating_cost(b):
        return (
            pyunits.convert(
                m.fs.zo_costing.total_fixed_operating_cost,
                to_units=pyunits.USD_2020 / pyunits.year,
            )
            + pyunits.convert(
                m.fs.zo_costing.total_variable_operating_cost,
                to_units=pyunits.USD_2020 / pyunits.year,
            )
            + pyunits.convert(
                m.fs.od_costing.total_operating_cost,
                to_units=pyunits.USD_2020 / pyunits.year,
            )
        )
    
    @m.fs.Expression(doc="Total cost of water recovery and brine/sludge disposed")
    def water_externalities(b):
        return pyunits.convert(
        m.fs.water_recovery_revenue
        - m.fs.brine_disposal_cost
        - m.fs.sludge_disposal_cost,
        to_units=pyunits.USD_2020 / pyunits.year,
        )
    
    @m.fs.Expression(doc="Total cost of Mg recovery and brine/sludge disposed")
    def Mg_externalities(b):
        return pyunits.convert(
        m.fs.Mg_recovery_revenue
        - m.fs.brine_disposal_cost
        - m.fs.sludge_disposal_cost,
        to_units=pyunits.USD_2020 / pyunits.year,
        )
    
    @m.fs.Expression(doc="Total cost of Li recovery and brine/sludge disposed")
    def Li_externalities(b):
        return pyunits.convert(
        m.fs.Li_recovery_revenue
        - m.fs.brine_disposal_cost
        - m.fs.sludge_disposal_cost,
        to_units=pyunits.USD_2020 / pyunits.year,
        )
    
    @m.fs.Expression(doc="Total cost of H2 and O2 recovery and brine/sludge disposed")
    def H2O2_externalities(b):
        return pyunits.convert(
            m.fs.H2_recovery_revenue
            + m.fs.O2_recovery_revenue
            - m.fs.brine_disposal_cost
            - m.fs.sludge_disposal_cost,
            to_units=pyunits.USD_2020 / pyunits.year,
        )
    
    @m.fs.Expression(doc="Total cost of Li, H2, and O2 recovery and brine/sludge disposed")
    def MgLiH2O2_externalities(b):
        return pyunits.convert(
            m.fs.Mg_recovery_revenue
            + m.fs.Li_recovery_revenue
            + m.fs.H2_recovery_revenue
            + m.fs.O2_recovery_revenue
            - m.fs.brine_disposal_cost
            - m.fs.sludge_disposal_cost,
            to_units=pyunits.USD_2020 / pyunits.year,
        )
    
    @m.fs.Expression(doc="Levelized cost of treatment with respect to volumetric feed flow (water only)")
    def LCOT_water(b): 
        return (
            b.total_capital_cost * b.zo_costing.capital_recovery_factor
            + b.total_operating_cost
            - b.water_externalities

        ) / (
            pyunits.convert(
                b.feed.properties[0].flow_vol,
                to_units=pyunits.m**3 / pyunits.year,
            )
            * b.zo_costing.utilization_factor
        )
    
    @m.fs.Expression(doc="Levelized cost of treatment with respect to volumetric feed flow (water and Li only)")
    def LCOT_MgLi(b): 
        return (
            b.total_capital_cost * b.zo_costing.capital_recovery_factor
            + b.total_operating_cost 
            - b.water_externalities
            - b.Li_externalities
            - b.Mg_externalities
        ) / (
            pyunits.convert(
                b.feed.properties[0].flow_vol,
                to_units=pyunits.m**3 / pyunits.year,
            )
            * b.zo_costing.utilization_factor
        )
    @m.fs.Expression(doc="Levelized cost of treatment with respect to volumetric feed flow (water, H2 and O2 only)")
    def LCOT_H2O2(b): 
        return (
            b.total_capital_cost * b.zo_costing_capital_recovery_factor
            + b.total_operating_cost
            - b.water_externalities
            - b.H2O2_externalities
        ) / (
            pyunits.convert(
                b.feed.properties[0].flow_vol,
                to_units=pyunits.m**3 / pyunits.year,
            )
            * b.zo_costing.utilization_factor
        )
    @m.fs.Expression(doc="Levelized cost of treatment with respect to volumetric feed flow (water, Li, H2 and O2)")
    def LCOT_MgLiH2O2(b): 
        return (
            b.total_capital_cost * b.zo_costing_capital_recovery_factor
            + b.total_operating_cost
            - b.water_externalities
            - b.Li_externalities
            - b.Mg_externalities
            - b.H2O2_externalities
        ) / (
            pyunits.convert(
                b.feed.properties[0].flow_vol,
                to_units=pyunits.m**3 / pyunits.year,
            )
            * b.zo_costing.utilization_factor
        )
    
    @m.fs.Expression(doc="Levelized cost of water with respect to volumetric permeate flow (water only)")
    def LCOW_water(b):
        return ( 
            b.total_capital_cost * b.zo_costing.capital_recovery_factor
            + b.total_operating_cost 
            + m.fs.brine_disposal_cost
            + m.fs.sludge_disposal_cost
        ) / (
            pyunits.convert(
                prtrt.gac.treated.properties[0].flow_vol,
                + desal.product.properties[0].flow_vol,
                + psttrt.IX.outlet.properties[0].flow_vol,
                to_units=pyunits.m**3 / pyunits.year,

            )
            * b.zo_costing.utilization_factor
        )
    @m.fs.Expression(doc="Levelized cost of water with respect to volumetric permeate flow (Li extraction)")
    def LCOW_MgLi(b):
        return (
            b.total_capital_cost * b.zo_costing.capital_recovery_factor
            + b.total_operating_cost
            + m.fs.brine_disposal_cost
            + m.fs.sludge_disposal_cost
        ) / (
            pyunits.convert(
                prtrt.gac.treated.properties[0].flow_vol,
                + Mgx.treated.properties[0].flow_vol,
                + Lix.treated.properties[0].flow_vol,
                + desal.product.properties[0].flow_vol,
                + psttrt.IX.outlet.properties[0].flow_vol,
                to_units=pyunits.m**3 / pyunits.year,
            )
            * b.zo_costing.utilization_factor
        )
    
    assert_units_consistent(m)

    #Set costing scalar factors
    iscale.set_scaling_factor(m.fs.zo_costing.total_capital_cost, 1e-4)
    iscale.set_scaling_factor(m.fs.od_costing.total_capital_cost, 1e-6)

    iscale.set_scaling_factor(m.fs.zo_costing.total_operating_cost, 1e-4)
    iscale.set_scaling_factor(m.fs.zo_costing.total_operating_cost, 1e-6)

    for block in m.fs.component_objects(Block, descend_into=True):
        if isinstance(block, UnitModelBlockData) and hasattr(block, "costing"):
            iscale.set_scaling_factor(block.costing.capital_cost, 1e-4)
    return
    
         
def initialize_costing(m):
        m.fs.zo_costing.initialize()
        m.fs.od_costing.initialize()

def display_costing(m):
    capex = value(pyunits.convert(m.fs.total_capital_cost, to_units=pyunits.MUSD_2020))

    prtrt_capex = value(
        pyunits.convert(
            m.fs.pretreatment.intake.costing.capital_cost,
            + m.fs.pretreatment.ferric_chloride_addition.costing.capital_cost,
            + m.fs.pretreatment.chlorination.costing.capital_cost,
            + m.fs.pretreatment.static_mixer.costing.capital_cost,
            + m.fs.pretreatment.storage.costing.capital_cost,
            + m.fs.pretreatment.screening.costing.capital_cost, 
            + m.fs.pretreatment.coag_and_floc.costing.capital_cost,
            + m.fs.pretreatment.sedimentation.costing.capital_cost,
            + m.fs.pretreatment.flotation.costing.capital_cost,
            + m.fs.pretreatment.gravity_basin.costing.capital_cost,
            + m.fs.pretreatment.mfiltration.costing.capital_cost,
            + m.fs.pretreatment.mbackwash_pump.costing.capital_cost,
            + m.fs.pretreatment.antiscalant_addition.costing.capital_cost,
            + m.fs.pretreatment.cfiltration.costing.capital_cost,
            + m.fs.pretreatment.gac.costing.capital_cost,
            + m.fs.pretreatment.gbackwash_pump.costing.capital_cost,
            + m.fs.pretreatment.mf_disposal.costing.capital_cost,
            + m.fs.pretreatment.gac_disposal.costing.capital_cost,
            to_units=pyunits.MUSD_2020,
        )
    )
    Mg_capex = value(
        pyunits.convert( 
            m.fs.Mgextraction.P1.costing.capital_cost,
            + m.fs.Mgextraction.S1.costing.capital_cost,
            + m.fs.Mgextraction.nano.costing.capital_cost,
            + m.fs.Mgstorage.storage.costing.capital_cost,
            + m.fs.Mgstorage.disposal.costing.capital_cost,
            + m.fs.Mgstorage.landfill.costing.capital_cost,
            to_units=pyunits.MUSD_2020,
        )
    )
    Li_capex = value(
        pyunits.convert( 
            m.fs.Liextraction.P1.costing.capital_cost,
            + m.fs.Liextraction.S1.costing.capital_cost,
            + m.fs.Liextraction.Elec.costing.capital_cost,
            + m.fs.Listorage.storage.costing.capital_cost,
            + m.fs.Listorage.disposal.costing.capital_cost,
            + m.fs.Listorage.landfill.costing.capital_cost,
            to_units=pyunits.MUSD_2020,
        )
    )
    desal_capex = value(
        pyunits.convert(
            m.fs.desalination.S0.costing.capital_cost,
            + m.fs.desalination.M0.costing.capital_cost,
            + m.fs.desalination.P0.costing.capital_cost,
            + m.fs.desalination.P1.costing.capital_cost,
            + m.fs.desalination.Elec.costing.capital_cost,
            + m.fs.desalination.S1.costing.capital_cost,
            + m.fs.desalination.ERD.costing_capital_cost,
            to_units=pyunits.MUSD_2020,
        )
    )
    
    psttrt_capex = value(
        pyunits.convert(
            m.fs.posttreatment.IX.costing.capital_cost,
            to_units=pyunits.USD_2020,
        )
    )

    if m.elec_type == "AWE":
        H2_capex = value(
            pyunits.convert(
                m.fs.H2extraction.ER.costing.capital_cost, 
                + m.fs.H2extraction.Compressor_O2.costing.capital_cost,
                + m.fs.H2extraction.Compressor_H2.costing.capital_cost,
                + m.fs.H2storage.O2.costing.capital_cost,
                + m.fs.H2storage.H2.costing.capital_cost,
                to_units=pyunits.MUSD_2020,              
            )
        )

    if m.elec_type == "PEM":
        H2_capex = value(
            pyunits.convert(
                m.fs.H2extraction.ER.costing.capital_cost,
                + m.fs.H2extraction.Compressor_O2.costing.capital_cost,
                + m.fs.H2extraction.Compressor_H2.costing.capital_cost,
                + m.fs.H2storage.O2.costing.capital_cost,
                + m.fs.H2storage.H2.costing.capital_cost,
                to_units=pyunits.MUSD_2020,
            )
        )
   
    opex = value(pyunits.convert(m.fs.total_operating_cost, to_units=pyunits.MUSD_2020 / pyunits.year))
    
    prtrt_opex = value(
        pyunits.convert(
            m.fs.pretreatment.intake.costing.total_operating_cost,
            + m.fs.pretreatment.ferric_chloride_addition.costing.total_operating_cost,
            + m.fs.pretreatment.chlorination.costing.total_operating_cost,
            + m.fs.pretreatment.static_mixer.costing.total_operating_cost,
            + m.fs.pretreatment.storage.costing.total_operating_cost,
            + m.fs.pretreatment.screening.costing.total_operating_cost,
            + m.fs.pretreatment.coag_and_flock.costing.total_operating_cost,
            + m.fs.pretreatment.sedimentation.costing.total_operating_cost,
            + m.fs.pretreatment.flotation.costing.total_operating_cost,
            + m.fs.pretreatment.gravity_basin.costing.total_operating_cost,
            + m.fs.pretreatment.mfiltration.costing.total_operating_cost,
            + m.fs.pretreatment.mbackwash_pump.costing.total_operating_cost,
            + m.fs.pretreatment.antiscalant_addition.costing.total_operating_cost,
            + m.fs.pretreatment.cfiltration.costing.total_operating_cost, 
            + m.fs.pretreatment.gac.costing.total_operating_cost, 
            + m.fs.pretreatment.gbackwash_pump.costing.total_operating_cost,
            + m.fs.pretreatment.mf_disposal.costing.total_operating_cost, 
            + m.fs.pretreatment.gac_disposal.costing.total_operating_cost, 
            to_units=pyunits.MUSD_2020 / pyunits.year,
        )
    )
    Mg_opex=  value(
        pyunits.convert( 
            m.fs.Mgextraction.P1.costing.total_operating_cost,
            + m.fs.Mgextraction.S1.costing.total_operating_cost,
            + m.fs.Mgextraction.nano.costing.total_operating_cost,
            + m.fs.Mgstorage.storage.costing.total_operating_cost,
            + m.fs.Mgstorage.disposal.costing.total_operating_cost,
            + m.fs.Mgstorage.landfill.costing.total_operating_cost,
            to_units=pyunits.MUSD_2020 / pyunits.year,
        )
    )

    Li_opex=  value(
        pyunits.convert( 
            m.fs.Liextraction.P1.costing.total_operating_cost,
            + m.fs.Liextraction.S1.costing.total_operating_cost,
            + m.fs.Liextraction.Elec.costing.total_operating_cost,
            + m.fs.Listorage.storage.costing.total_operating_cost,
            + m.fs.Listorage.disposal.costing.total_operating_cost,
            + m.fs.Listorage.landfill.costing.total_operating_cost,
            to_units=pyunits.MUSD_2020 / pyunits.year,
        )
    )

    desal_opex = value(
        pyunits.convert(
            m.fs.desalination.S0.costing.total_operating_cost,
            + m.fs.desalination.M0.costing.total_operating_cost,
            + m.fs.desalination.P0.costing.total_operating_cost,
            + m.fs.desalination.P1.costing.total_operating_cost,
            + m.fs.desalination.Elec.costing.total_operating_cost,
            + m.fs.desalination.S1.costing.total_operating_cost,
            + m.fs.desalination.ERD.costing.total_operating_cost,
            to_units=pyunits.MUSD_2020 / pyunits.year,
        )
    )


    if m.elec_type == "AWE":
        H2_opex = value(
            pyunits.convert(
                m.fs.H2extraction.ER.costing.total_operating_cost, 
                + m.fs.H2extraction.Compressor_O2.costing.total_operating_cost,
                + m.fs.H2extraction.Compressor_H2.costing.total_operating_cost,
                + m.fs.H2storage.O2.costing.total_operating_cost,
                + m.fs.H2storage.H2.costing.total_operating_cost,
                to_units=pyunits.MUSD_2020 / pyunits.year,              
            )
        )

    if m.elec_type == "PEM":
        H2_opex = value(
            pyunits.convert(
                m.fs.H2extraction.ER.costing.total_operating_cost,
                + m.fs.H2extraction.Compressor_O2.costing.total_operating_cost,
                + m.fs.H2extraction.Compressor_H2.costing.total_operating_cost,
                + m.fs.H2storage.O2.costing.total_operating_cost,
                + m.fs.H2storage.H2.costing.total_operating_cost,
                to_units=pyunits.MUSD_2020 / pyunits.year,
            )
        )

    psttrt_opex = value(
        pyunits.convert(
            m.fs.posttreatment.IX.costing.total_operating_cost,
            + m.fs.posttreatment.storage.costing.total_operating_cost,
            + m.fs.posttreatment.disposal.costing.total_operating_cost,
            + m.fs.posttreatment.landfill.costing.total_operating_cost,
            to_units=pyunits.MUSD_2020 / pyunits.year,
       )
    )

    H2O_prod_externalities = value(
        pyunits.convert(
            m.fs.water_externalities, to_units=pyunits.MUSD_2020 / pyunits.year
        )
    )

    H2OMgLi_prod_externalities = value(
        pyunits.convert(
            m.fs.MgLi_externalities, to_units=pyunits.MUSD_2020 / pyunits.year
        )
    )
    H2O2_prod_externalities = value(
        pyunits.convert(
            m.fs.H2O2_externalities, to_units=pyunits.MUSD_2020 / pyunits.year
        )
    )

    MgLiH2O2_prod_externalities = value(
        pyunits.convert(
            m.fs.MgLiH2O2_externalities, to_units=pyunits.MUSD_2020 / pyunits.year
        )
    )
    wrr = value(
        pyunits.convert(
            m.fs.water_recovery_revenue, to_units=pyunits.USD_2020 / pyunits.year
        )
    )
    Mgrr = value(
        pyunits.convert(
            m.fs.Mg_recovery_revenue, to_units=pyunits.USD_2020 / pyunits.year
        )
    )
    Lirr = value(
        pyunits.convert(
            m.fs.Li_recovery_revenue, to_units=pyunits.USD_2020 / pyunits.year
        )
    )
    H2rr = value(
        pyunits.convert(
            m.fs.H2_recovery_revenue, to_units=pyunits.USD_2020 / pyunits.year
        )
    )
    O2rr = value(
        pyunits.convert(
            m.fs.O2_recovery_revenue, to_units=pyunits.USD_2020 / pyunits.year
        )
    )
    sdc = value(
        pyunits.convert(m.fs.sludge_disposal_cost, to_units=pyunits.USD_2020 / pyunits.year)
    )

    bdc = value(
        pyunits.convert(m.fs.brine_disposal_cost, to_units=pyunits.USD_2020 / pyunits.year)
    )

    feed_flowrate = value(
        pyunits.convert(m.fs.feed.properties[0].flow_vol, to_units=m**3 / pyunits.hr)
    )

    capex_norm = (
        value(pyunits.convert(m.fs.total_capital_cost, to_units=pyunits.USD_2020))
        / feed_flowrate
    )

    annual_investment = value(
        pyunits.convert(
            m.fs.total_capital_cost*m.fs.zo_costing.capital_recovery_factor
            + m.fs.total_operating_cost,
            to_units=pyunits.USD_2020 / pyunits.year,
        )
    )

    opex_fraction = (
        100
        * value(
            pyunits.convert(
                m.fs.total_operating_cost, to_units=pyunits.USD_2020 / pyunits.year
            )
        )
        / annual_investment
    )

    lcot_water = value(pyunits.convert(m.fs.LCOT_water, to_units=pyunits.USD_2020 / pyunits.m**3))
    lcot_MgLi = value(pyunits.convert(m.fs.LCOT_Li, to_units=pyunits.USD_2020 / pyunits.m**3))
    lcot_h2o2 = value(pyunits.convert(m.fs.LCOT_H2O2, to_units=pyunits.USD_2020 / pyunits.m**3))
    lcot_MgLih2o2 = value(pyunits.convert(m.fs.LCOT_LiH2O2, to_units=pyunits.USD_2020 / pyunits.m**3))

    lcow_water = value(pyunits.convert(m.fs.LCOT_water, to_units=pyunits.USD_2020 / pyunits.m**3))
    lcow_MgLi = value(pyunits.convert(m.fs.LCOT_Li, to_units=pyunits.USD_2020 / pyunits.m**3))

    sec=m.fs.specific_energy_intensity()

    print("---Flow properties in feed, product, and disposal---")
    fp = {
        "Pretreatment Feed": [
            value(m.fs.feed.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.feed.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.feed.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.feed.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.feed.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.feed.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.feed.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
            )
        ],
        "Pretreatment Product": [
            value(m.fs.pretreatment.gac.treated.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.pretreatment.gac.treated.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.pretreatment.gac.treated.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.pretreatment.gac.treated.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.pretreatment.gac.treated.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.pretreatment.gac.treated.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.pretreatment.gac.treated.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                )
        ],
        "Pretreatment MF Disposal": [
            value(m.fs.pretreatment.mf_disposal.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.pretreatment.mf_disposal.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.pretreatment.mf_disposal.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.pretreatment.mf_disposal.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.pretreatment.mf_disposal.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.pretreatment.mf_disposal.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.pretreatment.mf_disposal.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                )
        ],
        "Pretreatment GAC Disposal": [
            value(m.fs.pretreatment.gac_disposal.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.pretreatment.gac_disposal.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.pretreatment.gac_disposal.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.pretreatment.gac_disposal.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.pretreatment.gac_disposal.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.pretreatment.gac_disposal.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.pretreatment.gac_disposal.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                )
        ],
        "Mg Extraction Produced H2O": [
            value(m.fs.Mgextraction.product.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.Mgextraction.product.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.Mgextraction.product.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.Mgextraction.product.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.Mgextraction.product.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.Mgextraction.product.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.Mgextraction.product.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                )
        ],
        "Mg Extraction Recovery": [
            value(m.fs.Mgextraction.byproduct.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.Mgextraction.byproduct.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]

                )
        ],
        "Li Extraction Produced H2O": [
            value(m.fs.Liextraction.treated.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.Liextraction.treated.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.Liextraction.treated.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.Liextraction.treated.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.Liextraction.treated.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.Liextraction.treated.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.Liextraction.treated.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                )
        ],
        "Li Extraction Product": [
            value(m.fs.Liextraction.product.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.Liextraction.product.properties[0].conc_mass_phase_comp["Liq", "Li_+"])
        ],
        "Desalination Product": [
            value(m.fs.desalination.product.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.desalination.product.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.desalination.product.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.desalination.product.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.desalination.product.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.desalination.product.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.desalination.product.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                )
        ],
        "Desalination Disposal": [
            value(m.fs.desalination.disposal.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.desalination.disposal.properties[0].conc_mass_phase_comp["Liq", "Na_+"]
                    + m.fs.desalination.disposal.properties[0].conc_mass_phase_comp["Liq", "Ca_2+"]
                    + m.fs.desalination.disposal.properties[0].conc_mass_phase_comp["Liq", "Li_+"]
                    + m.fs.desalination.disposal.properties[0].conc_mass_phase_comp["Liq", "Mg_2+"]
                    + m.fs.desalination.disposal.properties[0].conc_mass_phase_comp["Liq", "Cl_-"]
                    + m.fs.desalination.disposal.properties[0].conc_mass_phase_comp["Liq", "SO4_2-"]
                )
        ],
        "H2 Extraction Product H2": [
            value(m.fs.H2extraction.prod_H2.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.H2extraction.prod_H2.properties[0].conc_mass_phase_comp["Liq", "H2-v"])
        ],
        "H2 Extraction Product O2": [
            value(m.fs.H2extraction.prod_O2.properties[0].flow_vol_phase["Liq"]),
            value(m.fs.H2extraction.prod_O2.properties[0].conc_mass_phase_comp["Liq", "O2-v"])
        ],
    }
    fp_table = pd.DataFrame(
        data=fp,
        index=["Volumetric Flow Rate (m3/s)", "Total Ion Mass Concentration (kg/m3)"],
    )
    print(fp_table)

    print("---Performance Metrics---")

    pm_table = pd.DataFrame(
        data=[
            value(m.fs.Mgextraction.nano.area),
            value(m.fs.Mgextraction.nano.velocity[0]),
            value(m.fs.Mgextraction.nano.recovery_vol_phase[0, "Liq"]),
            value(m.fs.Mgextraction.nano.recovery_mass_phase_comp[0,"Liq","H2O"]),
            value(m.fs.Liextraction.Elec.voltage_applied[0]),
            value(m.fs.Liextraction.Elec.cell_pair_num),
            value(m.fs.Liextraction.Elec.mem_area),
            value(m.fs.Liextraction.Elec.recovery_vol_phase[0, "Liq"]),
            value(m.fs.Liextractgion.Elec.recovery_mass_phase_comp[0,"Liq","H2O"]),
            value(m.fs.desalination.Elec.voltage_applied[0]),
            value(m.fs.desalination.Elec.cell_pair_num),
            value(m.fs.desalination.Elec.mem_area),
            value(m.fs.desalination.Elec.recovery_vol_H2O[0]),
            value(m.fs.desalination.Elec.recovery_mass_phase_comp[0,"Liq","H2O"]),
            value(-1 * m.fs.desalination.ERD.work_mechanical[0].value /1e3),
            value(m.fs.H2extraction.ER.membrane_area),
            value(m.fs.H2extraction.ER.membrane_current_density[0]),
            value(m.fs.H2extraction.ER.voltage_cell[0]),
            value(m.fs.H2extraction.ER.voltage_efficiency[0]),
        ],
        columns=["value"],
        index=[
            "Mg2+ Nanofiltration Membrane Area, m2",
            "Mg2+ Nanofiltration Velocity, m/s",
            "Mg2+ Nanofiltration Water Recovery by Volume",
            "Li+ Electrodialysis Operation Voltage, V",
            "Li+ Electrodialysis Cell Pair Number",
            "Li+ Electrodialysis Membrane Area, m2",
            "Li+ Electrodialysis Water Recovery by Volume",
            "Desalination Operation Voltage, V",
            "Desalination Cell Pair Number",
            "Desalination Membrane Area, m2",
            "Desalination Water Recovery by Volume",
            "Desalination ERD Energy Recovery, kW"
            "Electrolyzer Membrane Area",
            "Electrolyzer Membrane Current Density, A/m2",
            "Electolyzer Operating Voltage, V",
            "Electrolyzer Voltage Effiency"
        ],
    )
    print(pm_table)

    print("---Pressure and Temperature Point Checking---")

    pt_dict = {

        "Pretreatment Feed": (
            value(m.fs.feed.outlet.pressure[0]),
            value(m.fs.feed.outlet.temperature[0]),
        ),

        "Magnesium Extraction - Nanofiltration Unit": (
            value(m.fs.Mgextraction.nano.inlet.pressure[0]),
            value(m.fs.Mgextraction.nano.inlet.temperature[0]),
        ),

        "Lithium Extraction - Electrodialysis": (
            value(m.fs.Liextraction.Elec.inlet_concentrate.pressure[0]),
            value(m.fs.Liextraction.Elec.inlet_concentrate.temperature[0]),
        ),

        "Electrodialysis RO Desalination": (
            value(m.fs.desalination.Elec.inlet_concentrate.pressure[0]),
            value(m.fs.desalination.Elec.inlet_concentrate.temperature[0]),
        ),

        "Posttreatment": (
            value(m.fs.posttreatment.IX.inlet.pressure[0]),
            value(m.fs.postreatment.IX.inlet.temperature[0]),
        ),

        "H2 Extraction": (
            value(m.fs.H2extraction.S1.inlet.pressure[0]),
            value(m.fs.H2extraction.S1.inlet.temperature[0]),
        )
    }
    pt_table = pd.Dataframe(data=pt_dict, index=["Pressue (Pa)", "Temperature (K)"])
    print(pt_table)




    print("\n System Costing Metrics:")
    print(f"\nTotal Capital Cost: {capex:.4f} M$")
    print(f"Pretreatment Capital Cost: {prtrt_capex:.4f} $")
    print(f"Magnesium Extraction Capital Cost: {Mg_capex: 4f} $")
    print(f"Lithium Extraction Capital Cost: {Li_capex: 4f} $")
    print(f"Desalination Capital Cost: {desal_capex: 4f} $")
    print(f"Ion Exchange Posttreatment Capital Cost: {psttrt_capex: 4f} $")
    print(f"Hydrogen & Oxygen Extraction Capital Cost: {H2_capex: 4f} $")

    print("\n--------------Unit Capital Costs----------------\n")
    for u in m.fs.zo_costing._registered_unit_costing:
        print(
            u.name,
            " : {price:0.3f} $".format(
                price=value(pyunits.convert(u.capital_cost, to_units=pyunits.USD_2020))
            )
        )
    for z in m.fs.od_costing._registered_unit_costing:
        print(
            z.name,
            " : {price:0.3f} $".format(
                price=value(pyunits.convert(z.capital_cost, to_units=pyunits.USD_2020))
            )
        )
    
    print(f"\nTotal Operating Cost: {opex:.4f} M$/year")
    print(f"Pretreatment Operational Cost: {prtrt_opex: .4f} M$/year")
    print(f"Magnesium Extraction Operational Cost: {Mg_opex: .4f} M$/year")
    print(f"Lithium Extraction Operational Cost: {Li_opex: .4f} M$/year")
    print(f"Desalination Operational Cost: {desal_opex: .4f} M$/year")
    print(f"Ion Exchange Posttreatment Operational Cost: {psttrt_opex: .4f} M$/year")
    print(f"Hydrogen & Oxygen Extraction Operational Cost: {H2_opex: .4f} M$/year")

    print(f"\nH2O Production Externalities: {H2O_prod_externalities: .4f} M$/year")
    print(f"H2O, Mg, and Li Production Externalities: {H2OMgLi_prod_externalities: .4f} M$/year")
    print(f"Hydrogen and Oxygen Production Externalities: {H2O2_prod_externalities: .4f} M$/year")
    print(f"Lithium, Hydrogen, & Oxygen Production Externalities: {MgLiH2O2_prod_externalities: 4f} M$/year")

    print(f"\nWater Recovery Revenue: {wrr:.4f} USD/year")
    print(f"Magnesium Recovery Revenue: {Mgrr: .4f} USD/year")
    print(f"Lithium Recovery Revenue: {Lirr: .4f} USD/year")
    print(f"Hydrogen Recovery Revenue: {H2rr: .4f} USD/year")
    print(f"Oxygen Recovery Revenue: {O2rr: .4f} USD/year")

    print(f"\nDesalination Brine Disposal Cost: {bdc: .4f} USD/year")
    print(f"Pretreatment Waste Disposal Cost: {sdc: .4f} USD/year")

    print(f"\nTotal Annual Cost: {annual_investment: .4f} $/year")
    print(f"Normalized Capital Cost: {capex_norm: .4f} $/m3feed/hr")
    print(f"Opex Fraction of Annual Cost: {opex_fraction: .4f} %")

    print(f"\nClean Water Production Levelized Cost of Treatment w/ Externalities: {lcot_water: .4f} $/m3 feed")
    print(f"Water & Li Production Levelized Cost of Treatment w/ Externalities: {lcot_MgLi: .4f} $/m3 feed")
    print(f"Hydrogen & Oxygen Production Levelized Cost of Treatment w/ Externalities: {lcot_h2o2: .4f} $/m3 feed")
    print(f"Li, Hydrogen & Oxygen Production Levelized Cost of Treatment w/ Externalities: {lcot_MgLih2o2: .4f} $/m3 feed")

    print(f"\nClean Water Production Levelized Cost of Water w/ Externalities: {lcow_water: .4f} $/m3 permeate")
    print(f"Li Production Levelized Cost of Water w/ Externalities: {lcow_MgLi: .4f} $/m3 permeate")

    print(f"\nSpecific Energy Intensity: {sec: .4f} kWh/m3 feed")

    
def export_to_ui():
        from watertap.ui.fsapi import FlowsheetInterface

        def noop(*args, **kwargs):
            return

        return FlowsheetInterface(
            name="Configuration 4",
            description="C4 Powered Softening & Li Extraction, Electrodialysis Desalination, & H2 Extraction From Seawater",
            do_export=noop,
            do_build=noop,
            do_solve=noop,
         )


if __name__ == "__main__":
        m = main(x_location="offshore", erd_type1="pressure_exchanger", erd_type2="pressure_exchanger", elec_type="PEM")



        
    

    


    

        



    











    




    



SyntaxError: invalid syntax (1105730436.py, line 401)